# Flow Matching for 1D Burgers' Control — `lab_four`

Companion to `lab_three.ipynb` (MIT 6.S184 lab 3: CFG-FM on MNIST).

This lab walks you through implementing Flow Matching for the **1D Burgers control problem** (DiffPhyCon paper Experiment 1), using your existing diffusion baselines from `trained_models/burgers/` as the comparison target.

## How to use this lab

1. Read the markdown cell at the top of each Part — it frames why this piece is needed and what's different from MNIST.
2. Find each `# Question N.M` cell and **fill in** the methods marked `raise NotImplementedError("Fill me in!")`. Each fill-in has a `# Step N:` comment above it that says exactly what to compute.
3. After each Part, **run the corresponding sanity check cell**.
4. When all sanity checks pass, run `part5_gamma_sweep` to reproduce the baseline comparison.

## Background reading (cross-referenced inline)

- `flow_matching_diffusion.md` — MIT FM theory (Prop 1 + Example 13)
- `notes_diffphycon_flow_bridge.md` — DDPM ↔ FM translation, §4 inpainting
- `notes_fm_prior_reweighting.md` — γ-reweighting math + code skeleton
- `notes_baseline_summary.md` — target numbers to beat
- `diffusion/diffusion_1d_burgers.py` — the DDPM reference impl this mirrors

## Data shape conventions

Throughout this file:

- `x` or `z` has shape `(b, 2, 16, 128)` — Burgers trajectory:
  - channel 0 = $u(t, x)$, the state field
  - channel 1 = $w(t, x)$, the control field
  - **time axis = 16**:
    - rows **0..10** = real physical time steps ($N_t = 11$, $t = 0, 1, \ldots, 10$)
    - rows **11..15** = **zero-padding** so the UNet's `dim_mults=(1,2,4,8)` can downsample 3 times ($16 \to 8 \to 4 \to 2$). No physical meaning.
    - w-channel row 10 is also 0 because $w$ drives transitions $t \to t{+}1$ and there's no transition out of $t = T$.
  - space axis = 128 ($N_x = 128$)
- `c` has shape `(b, 2, 128)` — boundary conditions:
  - `c[:, 0, :]` = $u_0$ (initial state = `x[:, 0, 0, :]`)
  - `c[:, 1, :]` = $u_T^*$ (target terminal = `x[:, 0, 10, :]`) — **row 10, not 15!** Row 10 is the last real time step; rows 11..15 are padding.
- `t` (FM time) has shape `(b, 1, 1, 1)` or scalar — $\tau \in [0, 1]$


In [1]:
# pyright: reportUnknownMemberType=false
from __future__ import annotations
import math
import os
import sys
from abc import ABC, abstractmethod
from typing import Tuple, Optional, Callable

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# Make project root importable so we can use existing diffphycon modules.
# Works both as a .py script (__file__ defined) and inside a Jupyter notebook
# cell (no __file__; fall back to CWD which should be the repo root).
try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Running inside Jupyter — assume CWD is the project root or flow/
    HERE = os.path.abspath(os.getcwd())
    if os.path.basename(HERE) != "flow":
        # CWD is repo root, point HERE at flow/
        HERE = os.path.join(HERE, "flow")
ROOT = os.path.dirname(HERE)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Part 0: Setup + Base Classes

These are the same abstractions you implemented/used in `lab_three.ipynb`. Copied here so this lab is self-contained.

**No fill-ins in Part 0** — just read through to remember the interfaces:

- `Sampleable` / `LabeledSampleable` — distributions you can sample from
- `ConditionalProbabilityPath` — defines $p_t(x \mid z)$
- `LinearAlpha` / `LinearBeta` — the schedule $\alpha_\tau = \tau,\; \beta_\tau = 1 - \tau$
- `GaussianConditionalProbabilityPath` — Gaussian path with this schedule
- `VectorFieldNet` — abstract velocity-field network
- `Trainer` — generic FM trainer (you'll subclass this)

In [4]:
class Sampleable(ABC):
    """A distribution we can sample from."""
    @abstractmethod
    def sample(self, num_samples: int) -> torch.Tensor:
        ...

In [5]:
class LabeledSampleable(ABC):
    """A joint distribution over (x, c) we can sample from.

    For Burgers: x = full trajectory (u, w), c = boundary condition (u_0, u_T*).
    Unlike MNIST (where c is a discrete class label), our c is a continuous vector.
    """
    @abstractmethod
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """Returns (x, c)."""
        ...

In [6]:
class ConditionalProbabilityPath(ABC):
    """p_t(x | z): a continuous interpolation from p_init to delta_z.

    Subclasses can implement two equivalent forms of `target_velocity`:
      - Form A (ε form): signature (x_t, z, t, eps)
      - Form B (x_t form, lab_three style): signature (x_t, z, t)

    The dispatcher `target_velocity(x_t, z, t, eps=None)` picks one.
    See markdown cell above for derivation.
    """
    @abstractmethod
    def sample_conditional_path(self, z: torch.Tensor, t: torch.Tensor):
        ...

    @abstractmethod
    def target_velocity(
        self,
        x_t: torch.Tensor,
        z: torch.Tensor,
        t: torch.Tensor,
        eps: torch.Tensor | None = None,
    ) -> torch.Tensor:
        ...

In [7]:
class LinearAlpha:
    """α_τ = τ. So α̇ = 1."""
    def __call__(self, t):
        return t

    def dt(self, t):
        return torch.ones_like(t)

In [8]:
class LinearBeta:
    """β_τ = 1 - τ. So β̇ = -1."""
    def __call__(self, t):
        return 1.0 - t

    def dt(self, t):
        return -torch.ones_like(t)

## Two equivalent forms of the target velocity $u^{\text{target}}(x_t \mid z)$

The next cell — `GaussianConditionalProbabilityPath` — implements the **conditional vector field** $u^{\text{target}}(x_t \mid z)$. There are **two mathematically equivalent ways** to write it. **They give identical numerical values** — the next cell implements both as `target_velocity_formA` and `target_velocity_formB`, with a dispatcher to switch between them. Read this to understand why.

### Setup

The conditional path is

$$
x_t \;=\; \alpha_t\, z \;+\; \beta_t\, \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)
$$

The conditional vector field is the time derivative of this flow.

### Form A — express in $\varepsilon$ (the direct form)

Differentiate the path directly:

$$
u^{\text{target}}(x_t \mid z) \;=\; \frac{d}{dt}\bigl[\alpha_t z + \beta_t \varepsilon\bigr] \;=\; \dot\alpha_t\, z + \dot\beta_t\, \varepsilon
$$

$$
\boxed{\;u^{\text{target}}(x_t \mid z) \;=\; \dot\alpha_t\, z \;+\; \dot\beta_t\, \varepsilon\;} \qquad \text{(Form A — ε form)}
$$

For our CondOT path ($\alpha_t = t,\;\beta_t = 1-t,\;\dot\alpha = 1,\;\dot\beta = -1$):

$$
u^{\text{target}} \;=\; z - \varepsilon
$$

**Signature**: `(x_t, z, t, eps)` — needs $\varepsilon$ (we already have it from sampling).

### Form B — express in $x_t$ alone (`lab_three` style)

Eliminate $\varepsilon$ by solving the path equation: $\varepsilon = (x_t - \alpha_t z)/\beta_t$. Substitute:

$$
\dot\alpha_t z + \dot\beta_t \cdot \frac{x_t - \alpha_t z}{\beta_t}
\;=\; \left(\dot\alpha_t - \frac{\dot\beta_t \alpha_t}{\beta_t}\right) z + \frac{\dot\beta_t}{\beta_t}\, x_t
$$

$$
\boxed{\;u^{\text{target}}(x_t \mid z) \;=\; \left(\dot\alpha_t - \frac{\dot\beta_t}{\beta_t}\alpha_t\right) z \;+\; \frac{\dot\beta_t}{\beta_t}\, x_t\;} \qquad \text{(Form B — $x_t$ form)}
$$

For our CondOT path:

$$
u^{\text{target}} \;=\; \frac{z - x_t}{1 - t}
$$

**Signature**: `(x_t, z, t)` — no $\varepsilon$ needed.

### They're identical (by construction)

Form B was derived FROM Form A by algebraic substitution. So they output the same number for the same `(z, x_t, t, ε)` consistent with $x_t = \alpha_t z + \beta_t \varepsilon$.

Run `sanity_check_two_forms()` (defined a few cells below) to verify: it samples $z$, $\varepsilon$, $x_t$ and asserts $\|u_A - u_B\|_\infty < 10^{-4}$.

### Trade-offs

| | Form A (ε form) | Form B ($x_t$ form, `lab_three`) |
|:---|:---|:---|
| **Signature** | `(x_t, z, t, eps)` | `(x_t, z, t)` |
| **Computation** | one multiply per term | one division by $\beta_t$ |
| **Numerical issue** | None | $\beta_t \to 0$ at $t \to 1$ ⇒ divide by zero |
| **Needs ε?** | Yes (have it from sampler) | No (uses $x_t$ instead) |
| **Consistency with `lab_three`** | ❌ different signature | ✅ same signature |
| **Pedagogical value** | Direct from path definition | Forces you to use the change-of-variable trick |

### Why does any of this matter?

**At inference** (sampling new data), we never compute $u^{\text{target}}$ — we call the **trained neural network** $u^\theta_t(x_t)$, which only takes $x_t$. So neither form's "advantage" actually applies to inference.

**At training**, both forms are valid CFM loss targets. Theorem 12 says the regression MSE has the same expectation either way; the per-batch values differ pointwise but in a way that averages out.

**Bottom line**: this is a notational choice. **Default is Form A** (ε form, direct derivative). If you want `lab_three` style exactly, change the dispatcher body to call `_formB`.

In [9]:
class GaussianConditionalProbabilityPath(ConditionalProbabilityPath):
    """p_t(x|z) = N(α_t z, β_t² I).

    Two equivalent forms of the conditional vector field are implemented:
        - `target_velocity_formA` : ε form (uses x_t, z, t, eps)
        - `target_velocity_formB` : x_t form (uses x_t, z, t)  — lab_three style

    See the markdown cell above ("Two equivalent forms of the target velocity")
    for the derivation. The `target_velocity` dispatcher below picks
    which form to use — swap the body to compare them.
    """
    def __init__(self, alpha: LinearAlpha, beta: LinearBeta):
        self.alpha = alpha
        self.beta = beta

    def sample_conditional_path(self, z, t):
        """x_t = α_t·z + β_t·ε,  ε ~ N(0, I).

        Returns (x_t, eps) so the caller can pass eps into Form A if they want.
        Form B doesn't need eps; just ignore the second return value.
        """
        eps = torch.randn_like(z)
        x_t = self.alpha(t) * z + self.beta(t) * eps
        return x_t, eps

    # -------- Form A: ε form --------
    def target_velocity_formA(self, x_t, z, t, eps):
        """u^target(x_t|z) = α̇_t · z + β̇_t · ε      (Form A, ε form)

        Direct derivative of the conditional flow ψ_t(x_0|z) = α_t·z + β_t·x_0.
        For our CondOT path (α=t, β=1-t, α̇=1, β̇=-1): u^target = z - ε.
        No division by β_t, so no t→1 singularity. Requires ε from sampler.
        """
        return self.alpha.dt(t) * z + self.beta.dt(t) * eps

    # -------- Form B: x_t form (lab_three style) --------
    def target_velocity_formB(self, x_t, z, t):
        """u^target(x_t|z) = (α̇ − β̇·α/β)·z + (β̇/β)·x_t     (Form B, x_t form)

        Obtained by substituting ε = (x_t − α·z)/β into Form A.
        For our CondOT path: u^target = (z − x_t) / (1 − t).
        Mathematically identical to Form A. Diverges at t = 1 (β → 0);
        callers should clamp t away from 1.
        """
        a   = self.alpha(t)
        b   = self.beta(t)
        da  = self.alpha.dt(t)
        db  = self.beta.dt(t)
        return (da - db * a / b) * z + (db / b) * x_t

    # -------- Dispatcher: which form is "active" --------
    # Change this body to switch between A and B. Both give identical results;
    # see `sanity_check_two_forms()` below to verify.
    def target_velocity(self, x_t, z, t, eps=None):
        """Default = Form A (ε form). Swap to Form B by editing this body.

        eps is required if using Form A; ignored if using Form B.
        """
        # === Active form (uncomment one) ===
        return self.target_velocity_formA(x_t, z, t, eps)
        # return self.target_velocity_formB(x_t, z, t)

In [10]:
def sanity_check_two_forms(seed: int = 0, tau: float = 0.5) -> float:
    """Verify Form A and Form B give numerically identical results.

    Returns max |Form A − Form B| over a random batch. Should be ≤ 1e-5 (float32).
    """
    torch.manual_seed(seed)
    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    z = torch.randn(4, 2, 16, 128)
    t = torch.full((4, 1, 1, 1), tau)
    x_t, eps = path.sample_conditional_path(z, t)
    u_A = path.target_velocity_formA(x_t, z, t, eps)
    u_B = path.target_velocity_formB(x_t, z, t)
    err = (u_A - u_B).abs().max().item()
    print(f"sanity_check_two_forms(tau={tau}): max |A - B| = {err:.2e}")
    if err < 1e-4:
        print("  ✅ Form A ≡ Form B  (as expected)")
    else:
        print("  ⚠️  Forms diverge by more than 1e-4 — bug somewhere.")
    return err

In [11]:
class VectorFieldNet(nn.Module, ABC):
    """Abstract: a network that predicts the velocity field u_t^θ(x | c)."""
    @abstractmethod
    def forward(self, x: torch.Tensor, t: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        ...

## How the `Trainer.train` loop works

The next cell defines the generic FM trainer. Most of it is obvious — `loss.backward()`, `opt.step()`, etc. — but the **printing logic** can look cryptic if you haven't seen this pattern. Let's walk through it.

### The two key lines

```python
self.loss_history.append(loss.item())
if step % print_every == 0:
    avg = float(np.mean(self.loss_history[-print_every:]))
    print(f"  step {step:6d}  loss={loss.item():.5f}  avg{print_every}={avg:.5f}")
```

### What `print_every` does

It's a **stride** — "print a progress line every $N$ steps". If `print_every = 100`, we print at step 0, 100, 200, 300, ... and stay quiet in between. This keeps the log readable when training for thousands of steps (you don't want a line per step).

### What `step % print_every == 0` means

`%` is the **modulo** operator (余数). `step % 100` is the remainder when `step` is divided by 100:

| `step` | `step % 100` | `step % 100 == 0`? |
|:---:|:---:|:---:|
| 0   | 0  | ✅ |
| 1   | 1  | ❌ |
| 50  | 50 | ❌ |
| 99  | 99 | ❌ |
| 100 | 0  | ✅ |
| 199 | 99 | ❌ |
| 200 | 0  | ✅ |

So `if step % print_every == 0:` is the idiomatic Python way to say "is `step` a multiple of `print_every`?". This fires exactly when we want a progress line.

### Why we print a rolling average, not the single-step loss

A single batch's loss is **noisy** — it depends on which random samples ended up in that batch. Looking at one number doesn't tell you much about whether training is actually working.

So we additionally compute:

```python
avg = float(np.mean(self.loss_history[-print_every:]))
```

`self.loss_history[-print_every:]` is **slice notation** that grabs the **last `print_every` entries** of the list (Python lets negative indices count from the end). Then `np.mean(...)` averages them.

So the output looks like:

```
step      0  loss=1.04832  avg100=1.04832
step    100  loss=0.83217  avg100=0.91408
step    200  loss=0.62113  avg100=0.74829
step    300  loss=0.51002  avg100=0.58217
```

The `avg100` column is what you actually watch — it's much smoother and shows the real trend. The single-step `loss` is just a sanity check that the latest batch isn't blowing up.

### How to tune `print_every`

- **Too small** (e.g. `print_every=1`): log floods, hard to spot trends
- **Too large** (e.g. `print_every=10000`): can't tell if training is broken until 10k steps in
- **Good default**: roughly $1/10$ to $1/50$ of `num_steps`. For `num_steps=500` (our smoke check), `print_every=50` is fine.

In [12]:
class Trainer(ABC):
    """Generic FM trainer skeleton — handles optimizer + step loop.
    Subclasses implement `get_train_loss(batch_size)`."""
    def __init__(self, net: nn.Module, lr: float = 1e-3):
        self.net = net
        self.opt = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.loss_history: list = []

    @abstractmethod
    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        ...

    def train(self, num_steps: int, batch_size: int = 64, print_every: int = 100):
        """Train for num_steps. Live tqdm progress bar with loss postfix.

        `print_every` is now the rolling-mean window for the smoothed "avg" loss
        shown in tqdm's postfix (no longer a print stride — tqdm self-throttles).
        """
        from tqdm.auto import tqdm

        self.net.train()
        pbar = tqdm(range(num_steps), desc="train", leave=True)
        for step in pbar:
            self.opt.zero_grad()
            loss = self.get_train_loss(batch_size)
            loss.backward()
            self.opt.step()
            self.loss_history.append(loss.item())
            window = min(print_every, len(self.loss_history))
            avg = float(np.mean(self.loss_history[-window:]))
            pbar.set_postfix({"loss": f"{loss.item():.4f}", f"avg{window}": f"{avg:.4f}"})
        return self.loss_history

In [13]:
# Burgers-specific imports (existing modules in the repo)
from dataset.data_1d import Burgers1D
from model.burgers_1d.unet import Unet2D
from diffusion.diffusion_1d_burgers import sigmoid_schedule_flip

/opt/homebrew/Caskroom/miniconda/base/envs/diffphycon/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/baochen/diffphycon/diffusion/diffusion_1d_burgers.py:730: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


# Part 1: Get a Feel for Burgers Data

Like lab_three Part 1 (where you visualized MNIST), we first look at the Burgers dataset before building any models.

A Burgers trajectory is a **2-channel "image"**:
- channel 0 = $u(t, x)$, the velocity field of the fluid
- channel 1 = $w(t, x)$, the external control force we applied
- time axis = 11 actual steps (padded to 16), space axis = 128 points

Conditioning $c = (u_0, u_T^*)$ = the initial state and target terminal state. We control $w(t, x)$ to drive the system from $u_0$ to $u_T^*$.

**No fill-ins in Part 1** — just helper functions for you to play with.

In [14]:
BURGERS_DATASET_NAME = "free_u_f_1e5_front_rear_quarter"  # → data/<name>/
T_IDX = 10  # row index of u(T*) inside the 16-step padded time axis (11 real steps, 0..10)

In [60]:
BURGERS_DATASET_NAME = "free_u_f_1e5_front_rear_quarter"
T_IDX = 10  # row index of u(T*) inside the 16-step padded time axis


def load_burgers_test(device: str = "cpu", dataset: str = BURGERS_DATASET_NAME) -> Burgers1D:
    """Same as load_burgers_train but loads the test split (held-out 2000 samples)."""
    return Burgers1D(
        dataset="burgers", input_steps=1, output_steps=10, time_interval=1,
        is_y_diff=False, split="test", transform=None, pre_transform=None,
        verbose=False, root_path=os.path.join(ROOT, "data", dataset),
        device=device, rescaler=10.0, stack_u_and_f=True,
        pad_for_2d_conv=True, partially_observed_fill_zero_unobserved=None, nt_total=11,
    )


def load_burgers_train(device: str = "cpu", dataset: str = BURGERS_DATASET_NAME) -> Burgers1D:
    """Load the Burgers training dataset (8000 samples)."""
    return Burgers1D(
        dataset="burgers", input_steps=1, output_steps=10, time_interval=1,
        is_y_diff=False, split="train", transform=None, pre_transform=None,
        verbose=False, root_path=os.path.join(ROOT, "data", dataset),
        device=device, rescaler=10.0, stack_u_and_f=True,
        pad_for_2d_conv=True, partially_observed_fill_zero_unobserved=None, nt_total=11,
    )

In [16]:
def visualize_trajectory(x: torch.Tensor, title: str = "", save_path: str | None = None):
    """Visualize one Burgers trajectory (or a small batch of them).

    Args:
        x: shape (2, 16, 128) or (b, 2, 16, 128)
    """
    import matplotlib.pyplot as plt
    if x.dim() == 3:
        x = x.unsqueeze(0)
    x = x.detach().cpu().numpy()
    b = x.shape[0]

    fig, axes = plt.subplots(b, 2, figsize=(12, 3 * b))
    if b == 1:
        axes = axes.reshape(1, -1)
    for i in range(b):
        # only show real time steps (0..10)
        axes[i, 0].imshow(x[i, 0, :11], aspect="auto", cmap="RdBu_r",
                          vmin=-x[i, 0, :11].max(), vmax=x[i, 0, :11].max())
        axes[i, 0].set_title(f"sample {i}: u(t, x)")
        axes[i, 0].set_xlabel("space"); axes[i, 0].set_ylabel("time")
        axes[i, 1].imshow(x[i, 1, :11], aspect="auto", cmap="PiYG",
                          vmin=-x[i, 1, :11].max(), vmax=x[i, 1, :11].max())
        axes[i, 1].set_title(f"sample {i}: w(t, x)  [control]")
        axes[i, 1].set_xlabel("space")
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        print(f"  saved {save_path}")
    else:
        plt.show()
    plt.close()

In [17]:
def visualize_noisy_samples(z: torch.Tensor, save_path: str | None = None):
    """Show how x_t = α_t·z + β_t·ε looks at several τ values.
    Builds intuition for what the FM model sees during training.
    """
    import matplotlib.pyplot as plt
    alpha, beta = LinearAlpha(), LinearBeta()
    taus = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    fig, axes = plt.subplots(1, len(taus), figsize=(3 * len(taus), 3))
    for i, tau in enumerate(taus):
        # tau is a Python float; PyTorch broadcasts a scalar against z's 4D shape.
        t = torch.tensor(tau)
        eps = torch.randn_like(z)
        x_t = alpha(t) * z + beta(t) * eps
        axes[i].imshow(x_t[0, 0, :11].detach().cpu().numpy(), aspect="auto", cmap="RdBu_r")
        axes[i].set_title(f"τ={tau}")
        axes[i].set_xticks([]); axes[i].set_yticks([])
    fig.suptitle("u-channel at increasing τ (noise → data)")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches="tight")
        print(f"  saved {save_path}")
    else:
        plt.show()
    plt.close()

# Part 2: Train the Joint FM Model

**Goal**: learn the velocity field $u_t^\theta(x \mid c)$ of the joint distribution $p(u, w \mid u_0, u_T^*)$ over Burgers trajectories.

**Three classes to fill in**:
- **Q2.1** — `BurgersDataset.sample` (sampling from $p_{\text{data}}$)
- **Q2.2** — `BurgersFlowTrainer.get_train_loss` (CFM loss + inpainting trick)
- **Q2.3** — `BurgersVectorField.forward` (Unet2D wrapped + $c$ injection)

**How Burgers differs from MNIST (lab_three Part 2)**:
- Conditioning $c$ is a **continuous vector** $(u_0, u_T^*)$, not a class index. So no `null label` and no CFG dropout ($\eta = 0$).
- We inject $c$ via **inpainting overwrite** (force `x[:, 0, 0, :] = u_0`, `x[:, 0, T_IDX, :] = u_T*`) instead of cross-attention or class embedding. This is the DiffPhyCon trick — see `notes_diffphycon_flow_bridge.md §4.3`.
- We train with an additional "inpainting trick" loss term: the target velocity at the boundary rows is **forced to 0**, teaching the model "if you see a clean boundary, don't change it". See `notes_diffphycon_flow_bridge.md §4.4`.

## Question 2.1 — `BurgersDataset.sample`

Build the data sampler. For each batch, return:
- `z` of shape `(b, 2, 16, 128)` — the full clean trajectory
- `c` of shape `(b, 2, 128)` — the boundary condition $(u_0, u_T^*)$, extracted from `z`'s u-channel rows 0 and 10

**2 fill-in Steps**.

### Breaking down `BurgersDataset.__init__`

Before you fill in `.sample()` (Q2.1), let's walk through what the constructor does. It looks like a lot but it's just **wrap a PyTorch Dataset + pre-load everything into memory for speed**.

```python
def __init__(self, dataset: Burgers1D, device: str = "cpu"):
    self.ds = dataset                                                      # 1
    self.device = device                                                   # 2
    all_z = torch.stack([self.ds[i] for i in range(len(self.ds))], dim=0)  # 3
    self.all_z = all_z.to(device)                                          # 4
    self.N = self.all_z.shape[0]                                           # 5
```

### Line 1 — `self.ds = dataset`

`dataset` is a `Burgers1D` instance — a PyTorch **Dataset** class. The thing to remember:

- `dataset[i]` returns one sample, a tensor of shape `(2, 16, 128)`
- `len(dataset)` returns the total number of samples (160 for our train set)

Mental model: a list-like object of `(2, 16, 128)` tensors that **lazily** reads from the HDF5 file on disk each time you access it.

### Line 2 — `self.device = device`

Stores a string like `"cpu"`, `"mps"`, or `"cuda"`. Used in Line 4.

### Line 3 — the list comprehension + `torch.stack`

This is two things glued together. Let's split:

**Part A — list comprehension**:
```python
[self.ds[i] for i in range(len(self.ds))]
```
is equivalent to:
```python
samples = []
for i in range(len(self.ds)):     # i = 0, 1, ..., 159
    samples.append(self.ds[i])    # each ds[i] has shape (2, 16, 128)
```
Result: a Python `list` of 160 tensors.

**Part B — `torch.stack`**:
```python
torch.stack(samples, dim=0)
```
takes the list of 160 tensors (each shape `(2, 16, 128)`) and stacks them along a **new dimension 0**:

| Input | Output |
|:---|:---|
| 160 tensors of shape `(2, 16, 128)` | 1 tensor of shape `(160, 2, 16, 128)` |

The first dim is now the "batch index" — `all_z[5]` gives you sample number 5.

> Quick distinction: `torch.stack` adds a new dim. `torch.cat` concatenates along an existing dim. If you tried `torch.cat(samples, dim=0)` here you'd get shape `(320, 16, 128)` — channels and batches glued together, which is **not** what we want.

### Line 4 — `.to(device)`

Moves the entire pre-loaded tensor to GPU/MPS if requested. On CPU this is a no-op.

### Line 5 — `self.N = self.all_z.shape[0]`

Stores 160 (the number of samples) so `.sample()` can pick valid indices in `[0, N)`.

### Why pre-stack instead of lazy-loading?

| Approach | Pros | Cons |
|:---|:---|:---|
| **Lazy** (call `self.ds[i]` each time in `sample`) | Low memory | Slow — each call reads HDF5 + applies transforms |
| **Pre-stack** (this lab) | Fast — `sample` becomes pure tensor indexing | Holds all data in RAM |

For Burgers (~5 MB total) pre-stacking is a clear win. For ImageNet-scale data you'd stay lazy and rely on PyTorch's `DataLoader` workers.

In [18]:
class BurgersDataset(LabeledSampleable):
    """Wraps the Burgers1D dataset, exposing (z, c) pairs.

    z: (b, 2, 16, 128)  full trajectory
    c:   (b, 2, 128)      (u_0, u_T*) — extracted from z itself

    The reason c lives separately even though it's "in" z: at training time
    the model sees noisy x_t (where boundary rows are also noised), so we keep
    the *clean* boundary in c and re-inject it via inpainting.
    """
    def __init__(self, dataset: Burgers1D, device: str = "cpu"):
        self.ds = dataset
        self.device = device
        # pre-stack to a single tensor for speed
        all_z = torch.stack([self.ds[i] for i in range(len(self.ds))], dim=0)
        self.all_z = all_z.to(device)
        self.N = self.all_z.shape[0]

    def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Returns:
            z: (b, 2, 16, 128) — full clean trajectory
            c:   (b, 2, 128)     — (u_0, u_T*) extracted from z's u-channel
        """
        # Step 1: Uniformly sample `num_samples` indices in [0, self.N), then use them
        #         to index self.all_z. The result is `z` with shape (num_samples, 2, 16, 128).
        #
        # Hint:
        #     idx = torch.randint(0, self.N, (num_samples,))   # shape (num_samples,) of ints
        #     z   = self.all_z[idx]                            # advanced indexing → (b, 2, 16, 128)
        #
        # Why this works: when you index a tensor with a LongTensor of indices, PyTorch
        # gathers the rows at those indices along dim 0. So all_z[[3, 7, 1]] is the same
        # as torch.stack([all_z[3], all_z[7], all_z[1]]), but in one vectorized call.
        idx = torch.randint(0, self.N, (num_samples,))
        z   = self.all_z[idx]     
        
        # Step 2: Extract c. Channel 0 is the u-field; rows 0 and T_IDX hold u_0 and u_T*.
        #         Stack them along a new "channel-of-condition" dim → shape (b, 2, 128).
        #
        # Hint: c = torch.stack([z[:, 0, 0, :], z[:, 0, T_IDX, :]], dim=1)
        c = torch.stack([z[:, 0, 0, :], z[:, 0, T_IDX, :]], dim=1)

        return z, c

## Question 2.2 — `BurgersFlowTrainer.get_train_loss`

This is **the heart of Flow Matching** — the conditional flow matching loss.

For each batch:
- sample $z \sim p_{\text{data}}$ and time $\tau \sim U[0, 1]$
- form noisy $x_\tau = \alpha_\tau z + \beta_\tau \epsilon$
- compute target velocity $u^{\text{target}}(x_\tau \mid z) = \dot\alpha_\tau z + \dot\beta_\tau \epsilon$
- regress: $\|u^\theta_\tau(x_\tau \mid c) - u^{\text{target}}(x_\tau \mid z)\|^2$

Plus the **inpainting trick** at Step 5: force target velocity to 0 at the boundary rows.

**6 fill-in Steps**. Compare to `lab_three.ipynb` Q2.2.

In [19]:
class BurgersFlowTrainer(Trainer):
    """Trains the joint Flow Matching model u_t^θ(x | c).

    Loss = ||u_t^θ(x_t | c) - u^target(x_t | z)||²  (averaged over t, z, ε).

    Inpainting trick (§4.4 in `notes_diffphycon_flow_bridge.md`):
        We also want the model to learn `if c is overwritten into x at row 0
        and row T, then your output at those rows should be 0` (= don't push
        the clean boundary anywhere). We enforce this by zeroing the *target*
        velocity at row 0 and row T_IDX in u-channel.
    """
    def __init__(
        self,
        net: VectorFieldNet,
        path: GaussianConditionalProbabilityPath,
        data: BurgersDataset,
        lr: float = 1e-3,
    ):
        super().__init__(net, lr=lr)
        self.path = path
        self.data = data

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        # Step 1: Sample a batch from self.data — get z (clean z) and c (boundary).
        #         Use self.data.sample(batch_size).
        z, c = self.data.sample(batch_size)

        # Step 2: Sample FM time t ~ Uniform[0, 1] with shape (b, 1, 1, 1) so it broadcasts
        #         against x of shape (b, 2, 16, 128). Use torch.rand.
        t = torch.rand(batch_size, 1, 1, 1, device=z.device)

        # Step 3: Sample x_t ~ p_t(x | z) via the path object.
        #         self.path.sample_conditional_path(z, t) returns (x_t, eps).
        x_t, eps = self.path.sample_conditional_path(z, t)

        # Step 4: Compute the target velocity via the path dispatcher:
        #             u_target = self.path.target_velocity(x_t, z, t, eps)
        #         The dispatcher calls Form A (ε form) by default; you can switch
        #         to Form B inside GaussianConditionalProbabilityPath to compare.
        #         (See "Two equivalent forms..." markdown cell above.)
        u_target = self.path.target_velocity(x_t, z, t, eps)

        # Step 5: Inpainting trick — force u_target to 0 at row 0 and row T_IDX,
        #         u-channel only.  See `notes_diffphycon_flow_bridge.md §4.4`.
        #         u_target[:, 0, 0, :]     = 0
        #         u_target[:, 0, T_IDX, :] = 0
        u_target[:, 0, 0, :]     = 0
        u_target[:, 0, T_IDX, :] = 0

        # Step 6: Forward through self.net (the BurgersVectorField) with (x_t, t, c),
        #         then compute MSE: ((u_pred - u_target) ** 2).mean()
        u_pred = self.net(x_t, t, c)
        
        return ((u_pred - u_target) ** 2).mean()

## Question 2.3 — `BurgersVectorField.forward`

Wraps `Unet2D` as a `VectorFieldNet`. The trick: **no embedding for $c$** — every forward call overwrites the boundary rows of $x$ with clean $c$ values before passing to the Unet.

This way train/inference are consistent: at both training time (Q2.2) and sampling time (Q3.2), the network sees clean boundaries at the same rows.

**3 fill-in Steps**.

In [20]:
class BurgersVectorField(VectorFieldNet):
    """Wraps Unet2D as a conditional velocity field.

    The trick: there is no embedding for c. Instead, every forward call
    *overwrites the boundary rows of x* with the clean c values before
    passing through the Unet. The Unet then operates as if its input
    already had the boundary baked in. This is exactly what we do at
    sampling time too (see Q3.1), so train/inference are consistent.

    See `notes_diffphycon_flow_bridge.md §4.3`.
    """
    def __init__(self, dim: int = 64, dim_mults: Tuple[int, ...] = (1, 2, 4, 8)):
        super().__init__()
        # channels=2 because input is (u, w)
        self.unet = Unet2D(dim=dim, dim_mults=dim_mults, channels=2)

    def forward(self, x: torch.Tensor, t: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (b, 2, 16, 128)  noisy trajectory at FM time t
            t: (b,) or (b, 1, 1, 1)  FM time
            c: (b, 2, 128)  boundary (u_0, u_T*)
        Returns:
            v: (b, 2, 16, 128)  predicted velocity field
        """
        # Step 1: Overwrite x with c at boundary rows.
        #         x_in = x.clone();  x_in[:, 0, 0, :] = c[:, 0]; x_in[:, 0, T_IDX, :] = c[:, 1]
        #         (this is the same logic as `inpaint_overwrite` in Q3.1 — see there for context)
        x_in = x.clone()
        x_in[:, 0, 0, :] = c[:, 0]
        x_in[:, 0, T_IDX, :] = c[:, 1]

        # Step 2: Unet2D.forward expects time as 1-D (b,). If t came in as (b, 1, 1, 1), flatten it.
        #         t_flat = t.view(-1).to(x_in.dtype)
        t_flat = t.view(-1).to(x_in.dtype)

        # Step 3: Call self.unet(x_in, t_flat) and return the result.
        return self.unet(x_in, t_flat)

# Part 3: Inpainting + ODE Sampling

**Goal**: sample $z$ from the trained velocity field.

In lab_three Part 3 you built a DiT transformer; here we don't need that — `Unet2D` is already capable. The novelty for Burgers is **how** we sample, specifically the inpainting overwrite that injects $(u_0, u_T^*)$.

**Three things to fill in**:
- **Q3.1** — `inpaint_overwrite` (helper)
- **Q3.2** — `BurgersEulerSampler.sample`
- **Q3.3** — sanity check (provided, no fill-in)

## Question 3.1 — `inpaint_overwrite`

A tiny helper called **before every Euler step** during sampling. Forces:
- `x[:, 0, 0, :]    = u_0`
- `x[:, 0, T_idx, :] = u_T*`

This is the DiffPhyCon way of injecting hard conditioning — see `notes_diffphycon_flow_bridge.md §4.3`. Different from RePaint (no re-noising); just clean overwrite.

**2 fill-in Steps**.

In [21]:
def inpaint_overwrite(x: torch.Tensor, c: torch.Tensor, T_idx: int = T_IDX) -> torch.Tensor:
    """Force x's u-channel boundary rows to equal c (clean values, no noise).

    Called *before every Euler step* during sampling. This is the DiffPhyCon
    way of injecting the hard conditioning constraint — see
    `notes_diffphycon_flow_bridge.md §4.3`.

    Args:
        x: (b, 2, 16, 128) current sample
        c: (b, 2, 128) where c[:, 0] = u_0 and c[:, 1] = u_T*
        T_idx: index of u_T* row (default 10)

    Returns:
        x with rows 0 and T_idx of channel 0 replaced by c.
    """
    x = x.clone()

    # Step 1: Overwrite x[:, 0, 0, :] with c[:, 0]  (the u_0 vector).
    x[:, 0, 0, :] = c[:, 0] 

    # Step 2: Overwrite x[:, 0, T_idx, :] with c[:, 1]  (the u_T* vector).
    x[:, 0, T_idx, :] = c[:, 1] 

    return x

## Question 3.2 — `BurgersEulerSampler.sample`

Euler ODE integration from $\tau = 0$ (noise) to $\tau = 1$ (clean data), with `inpaint_overwrite` before each step.

The flow ODE is $\dfrac{dx}{d\tau} = v_\tau^\theta(x \mid c)$, integrated with step size $d\tau = 1 / n_{\text{steps}}$.

**5 fill-in Steps**.

In [22]:
class BurgersEulerSampler:
    """Euler ODE sampler with inpainting overwrite.

    The flow ODE is:
        dx/dτ = v_τ^θ(x | c)
    We integrate from τ=0 (noise) to τ=1 (clean data) using Euler steps,
    with inpainting overwrite *before each step* to enforce the boundary.
    """
    def __init__(self, net: VectorFieldNet, n_steps: int = 100, tau_min: float = 1e-3):
        self.net = net
        self.n_steps = n_steps
        self.tau_min = tau_min   # avoid τ → 0 singularity (b_τ = 1/τ)

    @torch.no_grad()
    def sample(self, c: torch.Tensor, shape: Tuple[int, ...] = (2, 16, 128)) -> torch.Tensor:
        """
        Args:
            c: (b, 2, 128) boundary condition
            shape: per-sample shape, default (2, 16, 128)
        Returns:
            x_final: (b, *shape)
        """
        b = c.shape[0]
        device = c.device
        dtau = (1.0 - 2 * self.tau_min) / self.n_steps

        # Step 1: Initialize x with random noise of shape (b, *shape) on the correct device.
        #         x = torch.randn(b, *shape, device=device)
        x = torch.randn(b, *shape, device=device)

        # Step 2: Inpaint the very first time so x starts with the correct boundary.
        #         x = inpaint_overwrite(x, c)
        x = inpaint_overwrite(x, c)

        # Step 3: Euler loop — for step i in range(self.n_steps):
        #           tau = self.tau_min + i * dtau    (shape (b,) for the net)
        #           v   = self.net(x, tau_tensor, c)
        #           x   = x + v * dtau
        #           x   = inpaint_overwrite(x, c)
        #
        # Hint: build tau_tensor = torch.full((b,), tau, device=device).
        for i in range(self.n_steps):
            tau = self.tau_min + i * dtau  
            tau_tensor = torch.full((b,), tau, device=device)
            v   = self.net(x, tau_tensor, c)
            x   = x + v * dtau
            x   = inpaint_overwrite(x, c)


        # Step 5: Return x.
        return x

# Part 4: Prior Model + γ-Reweighting

This Part has **no parallel in lab_three** — it is the DiffPhyCon novelty.

We train a **second** model `net_prior` that learns the marginal $p(w \mid c)$ (i.e., the controls alone, no fluid dynamics). At sampling time we combine joint + prior to bias the velocity field toward ($\gamma > 1$) or away from ($\gamma < 1$) the prior, per the DiffPhyCon Eq. 9 reweighting.

You derived the FM-side formula yourself — see `notes_fm_prior_reweighting.md`:

$$\tilde{u}_\tau(x \mid c) = u_{\text{joint}}(x \mid c) + (\gamma - 1) \cdot \tilde{\eta}(\tau) \cdot \big[\,u_{\text{prior}}(x \mid c) - b_\tau \cdot [0, w]\,\big]$$

**Four things to fill in**:
- **Q4.1** — `BurgersPriorDataset` (u-channel zeroed in data)
- **Q4.2** — `BurgersPriorTrainer` (target velocity u-channel = 0)
- **Q4.3** — `w_scheduler_fm` (DDPM sigmoid_flip → FM time)
- **Q4.4** — `ReweightedVectorField` (the actual formula above)

## Question 4.1 — `BurgersPriorDataset.sample`

Same as `BurgersDataset.sample`, but the u-channel of $z$ is **zeroed out** before returning. The prior model only learns $p(w \mid c)$ — it shouldn't see fluid dynamics.

The DDPM equivalent is `diffusion_1d_burgers.py:400-402`.

**1 fill-in Step**.

In [23]:
class BurgersPriorDataset(BurgersDataset):
    """Dataset for training the prior p(w | c). Same as BurgersDataset but with u-channel zeroed.

    Why: the prior model should only see (and predict) w. By zeroing u in both
    input and output, we cleanly embed `u_prior` into the joint (u,w) space
    with u-block = 0 — matching the math in `notes_fm_prior_reweighting.md §2.4`.

    The DDPM equivalent is `diffusion_1d_burgers.py:400-402` (see comments there).
    """
    def sample(self, num_samples: int) -> Tuple[torch.Tensor, torch.Tensor]:
        z, c = super().sample(num_samples)

        # Step 1: Zero out the u-channel of z (channel 0). c is *not* zeroed —
        #         we still need it for inpainting and for the network's c input.
        #         z[:, 0] = 0
        z[:, 0] = 0

        return z, c

## Question 4.2 — `BurgersPriorTrainer` extras

Inherits from `BurgersFlowTrainer`, with **two additional zeroing steps**:
- Force the entire u-channel of `u_target` to 0 (not just rows 0 and T_IDX)
- Force the u-channel of `u_pred` to 0 (safety; the network shouldn't predict u dynamics)

This cleanly embeds $u_{\text{prior}}$ into the joint $(u, w)$ space with u-block = 0 — see `notes_fm_prior_reweighting.md §2.4`.

**2 fill-in Steps**.

---

### Design note — why not just set `out_dim=1` for the prior net?

Reasonable question: if we're going to zero the u-channel of `u_pred` anyway, why not build a smaller network with `out_dim=1` (only outputs the w-channel)?

**Two reasons we keep `out_dim=2`**:

1. **The "wasted" parameters are negligible** — measured exactly:
   - The only layer that differs is `final_conv = nn.Conv2d(dim, out_dim, 1)`:
     - `out_dim=2`: weight `(2, 64, 1, 1)` = 128 params + bias 2 = **130 params**
     - `out_dim=1`: weight `(1, 64, 1, 1)` = 64 params + bias 1 = **65 params**
     - Difference: **65 params**
   - Total Unet2D parameters: **35,707,906**
   - Waste ratio: $65 / 35{,}707{,}906 \approx 1.8 \times 10^{-6} = 0.0002\%$
   - Like 65 pixels out of a 1024×1024 image.

2. **The γ-reweighting formula stays clean** (see `notes_fm_prior_reweighting.md §3`):

   $$\tilde{u}_\tau(x \mid c) \;=\; u_{\text{joint}}(x \mid c) \;+\; (\gamma - 1)\,\tilde\eta(\tau)\,\big[\,u_{\text{prior}}(x \mid c) - b_\tau\,[0, w]\,\big]$$

   This formula assumes $u_{\text{joint}}$ and $u_{\text{prior}}$ have the **same shape** $(b, 2, 16, 128)$ so they add directly. If $u_{\text{prior}}$ were $(b, 1, 16, 128)$, every call to `ReweightedVectorField.forward` would need `torch.cat([zeros, v_prior], dim=1)` glue — adding code complexity for a 0.0002% saving.

The pattern of **"output same shape, mask the irrelevant channel"** is also exactly what the DDPM baseline does (`diffusion_1d_burgers.py:402`), so this keeps our FM comparison apples-to-apples.

In [24]:
class BurgersPriorTrainer(BurgersFlowTrainer):
    """Trains net_prior. Inherits from BurgersFlowTrainer; only difference is
    the target velocity for the u-channel must be forced to 0 (we don't want
    the prior model to learn anything about u dynamics)."""

    def get_train_loss(self, batch_size: int) -> torch.Tensor:
        # We can't easily call super().get_train_loss because we need to insert
        # the extra zeroing *between* computing target and computing loss. So we
        # repeat the Q2.2 steps inline here, with extras at Step 7 and Step 8.
        z, c = self.data.sample(batch_size)
        t = torch.rand(batch_size, 1, 1, 1, device=z.device)
        x_t, eps = self.path.sample_conditional_path(z, t)
        u_target = self.path.target_velocity(x_t, z, t, eps)
        u_target[:, 0, 0, :]     = 0
        u_target[:, 0, T_IDX, :] = 0

        # Step 7: Additionally zero the *entire* u-channel of u_target — not just
        #         rows 0 and T_IDX. We want the prior model to predict v=0 across
        #         all u-rows. See `notes_fm_prior_reweighting.md §2.4` (∇_u log p(w|c)=0).
        #
        #         u_target[:, 0] = 0
        u_target[:, 0] = 0
        # Forward + Step 8: zero the *output* u-channel before computing MSE.
        u_pred = self.net(x_t, t, c)

        # Step 8: u_pred[:, 0] = 0   (force the model output to also have u-block = 0)
        u_pred[:, 0] = 0

        loss = ((u_pred - u_target) ** 2).mean()
        return loss

## Question 4.3 — `w_scheduler_fm`

FM-time equivalent of DDPM's `sigmoid_schedule_flip`. The schedule is **small at τ=0** (noise end, weak reweighting) and **large at τ=1** (clean end, strong reweighting).

Map $\tau \to t_{\text{ddpm}} = \text{round}((1 - \tau) \cdot 999)$, then call existing `sigmoid_schedule_flip(t_ddpm)`.

**1 fill-in Step**.

### What is `w_scheduler_fm` and where does it fit?

`w_scheduler_fm(τ)` returns a single number $\tilde\eta(\tau) \in [0, 1]$ — the **time-dependent weight** in the γ-reweighting formula. It's not γ itself; it's a multiplier on $(\gamma - 1)$.

> ⚠️ **Naming convention heads-up** (read this if you've also looked at the Jellyfish experiment in this repo).
>
> Two papers in the DiffPhyCon family decompose the per-step reweighting strength differently:
>
> | Role | Jellyfish paper (L.1) | Burgers paper / this lab |
> |:---|:---|:---|
> | **Scalar knob** (you pick) | $\xi$ (`coeff_ratio_w`) | $\gamma$ (`prior_beta`); we use $(\gamma - 1)$ |
> | **Time function** | $\beta_{K-k}$ (hardcoded) | $\tilde\eta(\tau)$ (a schedule we can swap) |
> | **Combined per-step weight** | $\xi \cdot \beta_{K-k}$ | $(\gamma - 1) \cdot \tilde\eta(\tau)$ |
> | **What "γ" refers to** | the time-varying sequence $\gamma_k = 1 - \xi \beta_{K-k}$ | the scalar $\gamma$ |
>
> **Mathematically identical** — both formulas come out as `score_joint + (scalar × time-function) × score_prior`. The Jellyfish convention bundles the time variation INTO γ; the Burgers convention keeps γ scalar and exposes the schedule separately.
>
> **In this lab (lab_four)**: `gamma` is a Python scalar (you'll pass `gamma=0.3` or `gamma=2.5` to `ReweightedVectorField`). The time variation lives entirely in `w_scheduler_fm(τ)`. So when you read "γ is constant" in this lab, it's true *in this convention* — but the **effective** reweighting strength $(\gamma - 1) \cdot \tilde\eta(\tau)$ is still time-varying, exactly like Jellyfish's $\gamma_k$.

### Where it lives in the formula (`notes_fm_prior_reweighting.md §3 Step 5`)

$$
\boxed{\;\tilde u_\tau(x \mid c) \;=\; u_{\text{joint}}(x \mid c) \;+\; (\gamma - 1)\,\underbrace{\tilde\eta(\tau)}_{\text{this scheduler}}\,\big[\,u_{\text{prior}}(x \mid c) \;-\; b_\tau\,[0, w]\,\big]}
$$

So **γ alone is constant** (scalar knob you pick), and **η̃(τ) modulates how strongly γ acts at each time** in the ODE integration. Together they form the effective time-varying reweighting strength $(\gamma - 1)\cdot \tilde\eta(\tau)$.

### What it looks like

The function is borrowed from DDPM's `sigmoid_schedule_flip` (in `diffusion/diffusion_1d_burgers.py:110-111`) and remapped from DDPM time $t \in [0, 999]$ to FM time $\tau \in [0, 1]$ via:

$$
t_{\text{ddpm}} \;=\; \text{round}\big((1 - \tau) \cdot 999\big)
$$

so that $\tau = 0$ (noise end in FM) corresponds to $t = 999$ (noise end in DDPM), and $\tau = 1$ (clean end in FM) corresponds to $t = 0$ (clean end in DDPM).

Here's the actual curve (left = FM-time view, right = the original DDPM-time curve mirrored):

![η̃(τ) schedule plot](lab_four_eta_schedule.png)

Numerical values at a few representative τ:

| $\tau$ | $\tilde\eta(\tau)$ |
|:---:|:---:|
| 0.00 | 0.0003 |
| 0.30 | 0.0015 |
| 0.50 | 0.0033 |
| 0.70 | 0.0058 |
| 0.90 | 0.0127 |
| 0.99 | 0.0934 |
| 1.00 | 0.9990 |

So η̃ is **essentially zero** for most of integration (τ ∈ [0, 0.9]) and **suddenly ramps up** to ~1 right at the end (τ → 1).

### Why this shape — the "only reweight at the clean end" intuition

When integrating the ODE from $\tau = 0$ (pure noise) to $\tau = 1$ (clean data), the meaningful structure only emerges late. Reweighting the velocity field is essentially "biasing toward $p(w \mid c)$" — but **at the noise end, you don't yet know what your sample looks like**, so biasing toward the prior amplifies meaningless noise and destabilizes the trajectory.

By keeping η̃ ≈ 0 for the noisy portion and only "opening the gate" near the clean end, the reweighting fires when it can actually help: when $x_\tau$ is already a meaningful trajectory that just needs to be nudged closer to typical $w$.

**Empirical evidence** (`notes_baseline_summary.md §4.2`): without this scheduler (i.e., η̃ ≡ 1 across all τ), γ=0.3 gives $J = 0.0607$ — a disaster. With this scheduler, $J = 0.0083$ — basically as good as γ=1. **The scheduler is what makes off-baseline γ usable at all.**

### How to read the next cell

The fill-in is a single-line translation: take `τ`, convert to a DDPM step index via the formula above, look up `sigmoid_schedule_flip(t_ddpm)`. The dispatcher returns a torch tensor of the same shape as `τ` (scalar in, scalar out; 1-D in, 1-D out).

In [25]:
def w_scheduler_fm(tau: torch.Tensor) -> torch.Tensor:
    """FM-time equivalent of DDPM's `sigmoid_schedule_flip`.

    DDPM sigmoid_flip(t) is small at t→999 (noisy end) and large at t→0 (clean end).
    In FM time τ ∈ [0, 1]: τ=0 corresponds to DDPM t=999, τ=1 corresponds to t=0.

    See `notes_fm_prior_reweighting.md §3 Step 5`.

    Args:
        tau: (b,) or scalar in [0, 1]
    Returns:
        η̃(τ) — same shape as tau
    """
    # Step 1: Map τ to a DDPM step index, then call sigmoid_schedule_flip.
    #         For a scalar/1-D tau:
    #             t_ddpm = ((1.0 - tau) * 999).round().long().clamp(min=0, max=999)
    #             eta    = sigmoid_schedule_flip(t_ddpm)
    #         Note sigmoid_schedule_flip returns a torch tensor.
    orig_device = tau.device
    t_ddpm = ((1.0 - tau) * 999).round().long().clamp(min=0, max=999).cpu()
    eta    = sigmoid_schedule_flip(t_ddpm).float().to(orig_device)
    return eta

## Question 4.4 — `ReweightedVectorField.forward`

This is the **core of γ-reweighting in FM** — the formula you derived in `notes_fm_prior_reweighting.md §3`:

$$\boxed{\;\tilde{u}_\tau(x \mid c) = u_{\text{joint}}(x \mid c) + (\gamma - 1) \cdot \tilde{\eta}(\tau) \cdot \big[\,u_{\text{prior}}(x \mid c) - b_\tau \cdot [0, w]\,\big]\;}$$

where:
- $b_\tau = \dot\alpha_\tau / \alpha_\tau = 1/\tau$ (for the CondOT path $\alpha_\tau = \tau$)
- $\tilde{\eta}(\tau)$ from Q4.3
- $[0, w]$ = the $x$ vector with u-channel zeroed (only $w$ survives)

When $\gamma = 1$ the correction vanishes — sanity-checked in Q4.5.

**6 fill-in Steps**.

In [26]:
class ReweightedVectorField(VectorFieldNet):
    """The γ-reweighted velocity field:

        ũ_τ(x|c) = u_joint(x|c) + (γ-1) · η̃(τ) · [u_prior(x|c) - b_τ · [0, w]]

    where:
      - b_τ = α̇_τ / α_τ = 1/τ   (for CondOT path α_τ = τ)
      - η̃(τ) is the FM-time sigmoid_flip schedule from Q4.3
      - [0, w] = the x vector with u-channel zeroed (only w survives)

    Full derivation: `notes_fm_prior_reweighting.md §3`.

    When γ = 1 the correction term vanishes and this reduces to net_joint(x, t, c)
    exactly (sanity-checked in Q4.5).
    """
    def __init__(
        self,
        net_joint: VectorFieldNet,
        net_prior: VectorFieldNet,
        gamma: float = 1.0,
        use_scheduler: bool = True,
        tau_min: float = 1e-3,
    ):
        super().__init__()
        self.net_joint = net_joint
        self.net_prior = net_prior
        self.gamma = gamma
        self.use_scheduler = use_scheduler
        self.tau_min = tau_min

    def forward(self, x: torch.Tensor, t: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (b, 2, 16, 128)
            t: (b,)  in [tau_min, 1 - tau_min]
            c: (b, 2, 128)
        """
        # Step 1: Joint velocity. v_joint = self.net_joint(x, t, c)
        v_joint = self.net_joint(x, t, c)

        # Step 2: Build x_for_prior by zeroing u-channel of x.
        #         (Prior model was trained with u-channel always zero — Q4.1.)
        #         x_for_prior = x.clone();  x_for_prior[:, 0] = 0
        x_for_prior = x.clone()
        x_for_prior[:, 0] = 0

        # Step 3: Prior velocity. v_prior = self.net_prior(x_for_prior, t, c)
        #         Then force v_prior[:, 0] = 0 (safety — model output's u-block must be 0).
        v_prior = self.net_prior(x_for_prior, t, c)


        # Step 4: Build "[0, w]" — the x vector with u-channel zeroed (only w survives).
        #         x_w_only = x.clone();  x_w_only[:, 0] = 0
        #
        # Note: x_w_only ≠ x_for_prior in general — they happen to be equal here because
        # both zero out the u-channel. The two have different *meanings* though:
        # x_for_prior is what we feed the prior network; [0, w] is the argument inside the
        # `b_τ · [0, w]` correction term.  Same tensor, conceptually distinct.
        x_w_only = x.clone()
        x_w_only[:, 0] = 0

        # Step 5: Compute b_τ = 1/τ, clamped.  t may arrive as (b,) or (b,1,1,1).
        #         t_safe = t.clamp(min=self.tau_min)
        #         b_t = (1.0 / t_safe).view(-1, 1, 1, 1)   # broadcastable
        t_safe = t.clamp(min=self.tau_min)
        b_t = (1.0 / t_safe).view(-1, 1, 1, 1) 

        # Step 6: Apply the formula.
        #         eta = w_scheduler_fm(t.view(-1)) if self.use_scheduler else torch.ones_like(t.view(-1))
        #         eta = eta.view(-1, 1, 1, 1).to(x.dtype)
        #         correction = v_prior - b_t * x_w_only
        #         v_rw = v_joint + (self.gamma - 1) * eta * correction
        eta = w_scheduler_fm(t.view(-1)) if self.use_scheduler else torch.ones_like(t.view(-1))
        eta = eta.view(-1, 1, 1, 1).to(x.dtype)
        correction = v_prior - b_t * x_w_only
        v_rw = v_joint + (self.gamma - 1) * eta * correction
        
        return v_rw

# Part 5: γ Sweep + Baseline Comparison

This part has **no fill-ins** — once Parts 2-4 are working, this just plugs them together and runs the sweep that reproduces `notes_baseline_summary.md §3.1`.

We sweep $\gamma \in \{0.3, 0.5, 0.7, 0.9, 1.0, 1.5, 2.5\}$, compute $J$ and Energy, and compare directly against the existing DDPM baseline numbers.

In [27]:
# Baseline numbers from notes_baseline_summary.md §3.1 (DDPM, FOPC, with sigmoid_flip)
DDPM_BASELINE_FOPC = {
    0.3: {"J": 0.00830, "Energy": 1670.7},
    0.5: {"J": 0.00828, "Energy": 1666.0},
    0.7: {"J": 0.00825, "Energy": 1661.8},
    0.9: {"J": 0.00821, "Energy": 1658.0},
    1.0: {"J": 0.00820, "Energy": 1656.2},
    1.5: {"J": 0.00811, "Energy": 1648.1},
    2.5: {"J": 0.00796, "Energy": 1634.0},
}

In [28]:
def simulate_with_predicted_w(x_pred: torch.Tensor, c: torch.Tensor, rescaler: float = 10.0):
    """Run the PDE solver forward with predicted w and return the simulated u(t,x)."""
    from dataset.apps.generate_burgers import burgers_numeric_solve_free

    x_pred = x_pred.detach().cpu() * rescaler
    c_un   = c.detach().cpu()       * rescaler

    u_0 = c_un[:, 0]
    w   = x_pred[:, 1, :10, :].clone()

    Nx = w.shape[-1]
    w[:, :, Nx // 4 : (3 * Nx) // 4] = 0   # FOPC mask

    u_sim = burgers_numeric_solve_free(u_0, w, visc=0.01, T=1.0, dt=1e-4, num_t=10)
    return u_sim, w

In [29]:
def visualize_trajectory_with_simulation(
    x_pred: torch.Tensor,
    c: torch.Tensor,
    title: str = "",
    save_path: str | None = None,
    rescaler: float = 10.0,
):
    """5-panel viz (B+A enhanced):
    col 0: predicted u(t, x)     ┐ shared color scale
    col 1: simulated u(t, x)     ┘ (so col 0 vs col 1 visually comparable)
    col 2: diff = sim − pred     — highlights where model diverges from physics
    col 3: predicted w(t, x)     — FOPC dashed lines at x=0.25, 0.75
    col 4: terminal comparison   — u_0, u_T*, sim u(T), pred u(T) line plot
    """
    import matplotlib.pyplot as plt

    u_sim_t, w_used = simulate_with_predicted_w(x_pred, c, rescaler=rescaler)
    u_sim  = u_sim_t.numpy()
    u_pred = (x_pred[:, 0, :11].detach().cpu() * rescaler).numpy()
    w_pred = w_used.numpy()
    u_0    = (c[:, 0].detach().cpu() * rescaler).numpy()
    u_T    = (c[:, 1].detach().cpu() * rescaler).numpy()
    diff   = u_sim - u_pred

    b = x_pred.shape[0]
    nx = u_pred.shape[-1]
    x_axis = np.linspace(0, 1, nx)

    fig, axes = plt.subplots(b, 5, figsize=(22, 3.5 * b))
    if b == 1:
        axes = axes.reshape(1, -1)

    for row in range(b):
        u_vmax = max(abs(u_pred[row]).max(), abs(u_sim[row]).max(), 1e-6)

        ax = axes[row, 0]
        im0 = ax.imshow(u_pred[row], aspect="auto", origin="lower", extent=[0,1,0,1], cmap="RdBu_r", vmin=-u_vmax, vmax=u_vmax)
        ax.set_title(f"#{row}: predicted u(t,x)"); ax.set_xlabel("x"); ax.set_ylabel("t")
        plt.colorbar(im0, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[row, 1]
        im1 = ax.imshow(u_sim[row], aspect="auto", origin="lower", extent=[0,1,0,1], cmap="RdBu_r", vmin=-u_vmax, vmax=u_vmax)
        ax.set_title("simulated u(t,x)\n(PDE solve, same color ←)"); ax.set_xlabel("x"); ax.set_ylabel("t")
        plt.colorbar(im1, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[row, 2]
        diff_vmax = max(abs(diff[row]).max(), 1e-6)
        im2 = ax.imshow(diff[row], aspect="auto", origin="lower", extent=[0,1,0,1], cmap="seismic", vmin=-diff_vmax, vmax=diff_vmax)
        ax.set_title(f"diff = sim − pred\nmax|err| = {diff_vmax:.3f}"); ax.set_xlabel("x"); ax.set_ylabel("t")
        plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[row, 3]
        w_vmax = max(abs(w_pred[row]).max(), 1e-6)
        im3 = ax.imshow(w_pred[row], aspect="auto", origin="lower", extent=[0,1,0,1], cmap="PiYG", vmin=-w_vmax, vmax=w_vmax)
        ax.set_title("predicted w(t,x)\n(middle 1/2 = 0 — FOPC)"); ax.set_xlabel("x"); ax.set_ylabel("t")
        ax.axvline(0.25, color="k", lw=0.6, ls="--")
        ax.axvline(0.75, color="k", lw=0.6, ls="--")
        plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[row, 4]
        ax.plot(x_axis, u_0[row],        label="$u_0$",            color="gray", lw=1)
        ax.plot(x_axis, u_T[row],        label="$u_T^*$ (target)", color="k",    lw=2)
        ax.plot(x_axis, u_sim[row, -1],  label="sim $u(T)$",       color="C0",   ls="--")
        ax.plot(x_axis, u_pred[row, 10], label="pred $u(T)$",      color="C1",   ls=":")
        ax.set_title("terminal-state comparison")
        ax.set_xlabel("x"); ax.set_ylabel("u")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=110, bbox_inches="tight")
        print(f"  saved {save_path}")
    else:
        plt.show()
    plt.close()

In [30]:
def compute_J_and_energy(
    x_pred: torch.Tensor,
    c: torch.Tensor,
    rescaler: float = 10.0,
) -> Tuple[float, float]:
    """Compute J = ||u_sim(T) - u_T*||² and Energy = ||w||² for a single prediction batch.

    Now with FOPC partial-control alignment: w is zeroed in the middle 50% of space
    before PDE simulation (matches DDPM baseline).
    """
    from utils import burgers_metric

    b = x_pred.shape[0]
    x_pred = x_pred.detach().cpu() * rescaler
    c_un   = c.detach().cpu()       * rescaler

    Nt = 11
    Nx = c_un.shape[-1]
    u_target = torch.zeros(b, Nt, Nx)
    u_target[:, 0]  = c_un[:, 0]
    u_target[:, 10] = c_un[:, 1]

    w = x_pred[:, 1, :10, :]

    J_per_sample, E_per_sample = burgers_metric(
        u_target, w, target="final_u",
        partial_control="front_rear_quarter",
    )
    return float(J_per_sample.mean()), float(E_per_sample.mean())

## Part 5 helpers — train net_joint and net_prior at paper-aligned scale

The two cells below define `train_joint_for_part5` and `train_prior_for_part5` — wrappers that build a `BurgersVectorField` with the **same architecture as the DDPM baseline** (`dim=64`, `dim_mults=(1, 2, 4, 8)`, `lr=1e-4`, `batch=64`, dataset = full 10k `free_u_f_1e4_front_rear_quarter`).

**Default `num_steps` is the small-scale verify value** (joint=5000, prior=1500 — roughly 1/5 of paper-baseline). Once you confirm J drops to a sensible range, bump to the full numbers (25000 + 6250) for paper-grade comparison.

After defining + training, pass the two nets to `part5_gamma_sweep(net_joint, net_prior)` to see the 7-γ table vs DDPM baseline.

In [31]:
def plot_loss_history(
    losses,
    save_path: Optional[str] = None,
    title: str = "Training loss",
    window: int = 100,
):
    """Plot a loss curve from a list/array OR from a .pt checkpoint path."""
    import matplotlib.pyplot as plt

    if isinstance(losses, (str, os.PathLike)):
        ckpt = torch.load(losses, map_location="cpu", weights_only=False)
        if isinstance(ckpt, dict) and "loss_history" in ckpt:
            losses = ckpt["loss_history"]
        else:
            raise ValueError(f"{losses} doesn't contain a 'loss_history' key")
    losses = np.asarray(losses, dtype=float)
    if losses.size == 0:
        print("  (empty loss history)"); return

    w = min(window, losses.size)
    kernel = np.ones(w) / w
    smoothed = np.convolve(losses, kernel, mode="valid")
    smoothed_x = np.arange(w - 1, losses.size)

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.plot(losses, color="lightsteelblue", lw=0.5, alpha=0.6, label="per-step")
    ax.plot(smoothed_x, smoothed, color="navy", lw=1.5, label=f"rolling mean (w={w})")
    ax.set_yscale("log")
    ax.set_xlabel("training step")
    ax.set_ylabel("loss (log scale)")
    ax.set_title(f"{title}  (n={losses.size}, final smoothed = {smoothed[-1]:.4f})")
    ax.grid(alpha=0.3, which="both")
    ax.legend()
    plt.tight_layout()

    if save_path:
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=110, bbox_inches="tight")
        print(f"  saved loss plot: {save_path}")
    else:
        plt.show()
    plt.close()


def _train_with_checkpoints(
    trainer, num_steps, batch_size, save_path, checkpoint_every, print_every=200,
):
    if save_path is None or checkpoint_every <= 0 or checkpoint_every >= num_steps:
        trainer.train(num_steps=num_steps, batch_size=batch_size, print_every=print_every)
        return

    base, ext = os.path.splitext(save_path)
    loss_plot_path = f"{base}_losses.png"
    os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)

    done = 0
    while done < num_steps:
        chunk = min(checkpoint_every, num_steps - done)
        trainer.train(num_steps=chunk, batch_size=batch_size, print_every=print_every)
        done += chunk
        ckpt = {"state_dict": trainer.net.state_dict(), "loss_history": list(trainer.loss_history), "step": done}
        torch.save(ckpt, save_path)
        torch.save(ckpt, f"{base}_step{done}{ext}")
        plot_loss_history(trainer.loss_history, save_path=loss_plot_path,
                          title=f"{os.path.basename(base)}  (step {done}/{num_steps})")
        print(f"  [checkpoint] step {done}/{num_steps}  →  {base}_step{done}{ext}")

In [32]:
def train_joint_for_part5(
    num_steps: int = 5000,
    batch_size: int = 64,
    lr: float = 1e-4,
    save_path: Optional[str] = None,
    checkpoint_every: int = 1000,
    device: Optional[str] = None,
):
    """Joint p(u,w|c) FM training. Periodic checkpoint + loss plot."""
    device = device or ("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"  device: {device}")

    ds_full = load_burgers_train(device=device, dataset="free_u_f_1e4_front_rear_quarter")
    ds = BurgersDataset(ds_full, device=device)
    print(f"  dataset size: {ds.N} samples (vs sanity's 160)")

    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    net = BurgersVectorField(dim=64, dim_mults=(1, 2, 4, 8)).to(device)
    print(f"  net params: {sum(p.numel() for p in net.parameters()):,}")
    print(f"  training {num_steps} steps (checkpoint every {checkpoint_every})...")

    trainer = BurgersFlowTrainer(net, path, ds, lr=lr)
    _train_with_checkpoints(trainer, num_steps, batch_size, save_path, checkpoint_every)
    print(f"  done. final loss (smoothed): {float(np.mean(trainer.loss_history[-100:])):.5f}")
    return net

In [33]:
def train_prior_for_part5(
    num_steps: int = 1500,
    batch_size: int = 64,
    lr: float = 1e-4,
    save_path: Optional[str] = None,
    checkpoint_every: int = 500,
    device: Optional[str] = None,
):
    """Prior p(w|c) FM training. Periodic checkpoint + loss plot."""
    device = device or ("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"  device: {device}")

    ds_full = load_burgers_train(device=device, dataset="free_u_f_1e4_front_rear_quarter")
    ds = BurgersPriorDataset(ds_full, device=device)
    print(f"  dataset size: {ds.N} samples")

    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    net = BurgersVectorField(dim=64, dim_mults=(1, 2, 4, 8)).to(device)
    print(f"  net params: {sum(p.numel() for p in net.parameters()):,}")
    print(f"  training {num_steps} steps (prior — u-channel = 0; ckpt every {checkpoint_every})...")

    trainer = BurgersPriorTrainer(net, path, ds, lr=lr)
    _train_with_checkpoints(trainer, num_steps, batch_size, save_path, checkpoint_every)
    print(f"  done. final loss (smoothed): {float(np.mean(trainer.loss_history[-100:])):.5f}")
    return net

In [67]:
class EMA:
    """Maintains an exponential moving average of model parameters.
    
    Standard DDPM choice: decay=0.995, effective averaging window ~200 steps.
    Smooths out gradient-noise wobble → cleaner sampling outputs.
    """
    def __init__(self, model: nn.Module, decay: float = 0.995):
        self.decay = decay
        self.shadow = {name: p.detach().clone() for name, p in model.named_parameters()}

    @torch.no_grad()
    def update(self, model: nn.Module):
        for name, p in model.named_parameters():
            self.shadow[name].mul_(self.decay).add_(p.data, alpha=1.0 - self.decay)

    @torch.no_grad()
    def copy_to(self, model: nn.Module):
        for name, p in model.named_parameters():
            p.data.copy_(self.shadow[name])

In [ ]:
def finetune_with_ema(
    net: nn.Module,
    trainer_class: type,      # BurgersFlowTrainer (joint) or BurgersPriorTrainer
    dataset,                  # matching BurgersDataset or BurgersPriorDataset
    num_steps: int = 2000,
    batch_size: int = 64,
    lr: float = 1e-4,
    ema_decay: float = 0.995,
    save_path_ema: Optional[str] = None,
):
    """Continue training `net` for `num_steps` while building EMA shadow.

    Returns a NEW net with EMA-smoothed weights (original `net` is also
    trained in-place, so caller has both for comparison).
    """
    import copy
    from tqdm.auto import tqdm

    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    trainer = trainer_class(net, path, dataset, lr=lr)
    ema = EMA(net, decay=ema_decay)

    print(f"  fine-tuning {num_steps} steps with EMA decay={ema_decay}...")
    pbar = tqdm(range(num_steps), desc="finetune+EMA")
    net.train()
    for step in pbar:
        trainer.opt.zero_grad()
        loss = trainer.get_train_loss(batch_size)
        loss.backward()
        trainer.opt.step()
        trainer.loss_history.append(loss.item())
        ema.update(net)
        window = min(100, len(trainer.loss_history))
        avg = float(np.mean(trainer.loss_history[-window:]))
        pbar.set_postfix({"loss": f"{loss.item():.4f}", f"avg{window}": f"{avg:.4f}"})

    net_ema = copy.deepcopy(net).eval()
    ema.copy_to(net_ema)

    if save_path_ema:
        os.makedirs(os.path.dirname(save_path_ema) or ".", exist_ok=True)
        torch.save({"state_dict": net_ema.state_dict()}, save_path_ema)
        print(f"  saved EMA-weighted net: {save_path_ema}")

    print(f"  done. final loss (smoothed): {float(np.mean(trainer.loss_history[-100:])):.5f}")
    return net_ema

In [82]:
def inference_and_plot(
    net_joint: VectorFieldNet,
    net_prior: Optional[VectorFieldNet] = None,
    gamma: float = 1.0,
    n_samples: int = 3,
    n_steps: int = 100,
    use_scheduler: bool = True,
    save_path: Optional[str] = None,
    device: Optional[str] = None,
    dataset_name: str = "free_u_f_1e4_front_rear_quarter",
    split: str = "train",
    seed: Optional[int] = None,
):
    """Run inference on trained net(s) → produce 5-panel viz. Returns (x_pred, c, J, E).
    
    seed: if set, fixes BOTH the boundary c draw AND the sampler's initial x_0
          (since torch RNG is global). Use this to compare across n_steps fairly.
    """
    device = device or ("mps" if torch.backends.mps.is_available() else "cpu")
    if save_path is None:
        save_path = os.path.join(HERE, "lab_four_inference.png")

    loader = load_burgers_train if split == "train" else load_burgers_test
    ds_full = loader(device=device, dataset=dataset_name)
    ds = BurgersDataset(ds_full, device=device)
    if seed is not None:
        torch.manual_seed(seed)
    _, c = ds.sample(n_samples)
    print(f"  dataset split: {split} ({ds.N} samples), seed={seed}")

    if net_prior is None or gamma == 1.0:
        print(f"  inference: joint only (γ=1, no reweighting)")
        net = net_joint
    else:
        print(f"  inference: reweighted with γ={gamma}, scheduler={use_scheduler}")
        net = ReweightedVectorField(
            net_joint, net_prior, gamma=gamma, use_scheduler=use_scheduler,
        ).to(device)

    net.eval()
    sampler = BurgersEulerSampler(net, n_steps=n_steps)
    x_pred = sampler.sample(c)
    print(f"  sampled {n_samples} trajectories, shape={tuple(x_pred.shape)}")

    J, E = compute_J_and_energy(x_pred, c)
    baseline_J, baseline_E = 0.0082, 1656
    print(f"  J      = {J:.4f}   (DDPM baseline γ=1: {baseline_J})  →  {J/baseline_J:.1f}x")
    print(f"  Energy = {E:.1f}     (DDPM baseline γ=1: {baseline_E})")

    title = f"Inference: γ={gamma}, {n_samples} samples, {n_steps}-step Euler, seed={seed}"
    visualize_trajectory_with_simulation(x_pred, c, title=title, save_path=save_path)

    return x_pred, c, J, E

In [83]:
def compare_n_steps_visually(
    net_joint: VectorFieldNet,
    net_prior: Optional[VectorFieldNet] = None,
    gamma: float = 2.5,
    n_steps_list: list = [100, 500, 1000],
    seed: int = 42,
    n_samples: int = 3,
    save_dir: Optional[str] = None,
):
    """Loop inference_and_plot over n_steps with shared seed → N PNGs for visual comparison.

    Because seed is fixed, all calls use the same boundary c and same initial
    noise x_0 — only n_steps differs, so visual diff is purely ODE precision.
    """
    if save_dir is None:
        save_dir = os.path.join(ROOT, "flow")
    print(f"\n=== compare_n_steps_visually: γ={gamma}, seed={seed} ===")
    results = []
    for n in n_steps_list:
        save_path = os.path.join(save_dir, f"lab_four_inf_gamma{gamma}_n{n}_seed{seed}.png")
        print(f"\n--- n_steps={n} ---")
        _, _, J, E = inference_and_plot(
            net_joint, net_prior,
            gamma=gamma, n_samples=n_samples, n_steps=n,
            save_path=save_path, seed=seed,
        )
        results.append({"n_steps": n, "J": J, "Energy": E, "path": save_path})
    print(f"\n--- summary ---")
    print(f"{'n_steps':>8s} | {'J':>10s} | {'Energy':>10s}")
    for r in results:
        print(f"{r['n_steps']:>8d} | {r['J']:>10.5f} | {r['Energy']:>10.1f}")
    return results

In [84]:
def part5_gamma_sweep(
    net_joint: VectorFieldNet,
    net_prior: VectorFieldNet,
    n_samples: int = 8,
    n_steps: int = 100,
    device: Optional[str] = None,
    dataset_name: str = "free_u_f_1e4_front_rear_quarter",
    split: str = "train",
    seed: Optional[int] = None,
    save_path: Optional[str] = None,
):
    """Sweep γ ∈ {0.3, 0.5, 0.7, 0.9, 1.0, 1.5, 2.5} and compare to DDPM baseline.

    Auto-detects device from net_joint if device=None.
    `split="test"` uses held-out 2000 test samples.
    `seed` (int): fixes which n_samples boundary pairs get drawn.
    `save_path`: where to save the gamma-sweep PNG. Defaults to flow/lab_four_gamma_sweep.png
                 (gets overwritten across runs — pass distinct names to keep multiple).
    """
    if device is None:
        device = str(next(net_joint.parameters()).device)
    loader = load_burgers_train if split == "train" else load_burgers_test
    ds = BurgersDataset(loader(device=device, dataset=dataset_name), device=device)
    if seed is not None:
        torch.manual_seed(seed)
    _, c = ds.sample(n_samples)
    print(f"  using split={split!r} ({ds.N} samples available), seed={seed}")

    gammas = [0.3, 0.5, 0.7, 0.9, 1.0, 1.5, 2.5]
    results = {}

    import time
    from tqdm.auto import tqdm

    print("\n" + "=" * 92)
    print(f"{'γ':>5s} | {'FM J':>10s} | {'DDPM J':>10s} | {'FM Energy':>10s} | {'DDPM Energy':>12s} | {'wall (s)':>9s}")
    print("-" * 92)

    pbar = tqdm(gammas, desc="γ sweep")
    for g in pbar:
        t0 = time.time()
        rw = ReweightedVectorField(net_joint, net_prior, gamma=g, use_scheduler=True).to(device)
        sampler = BurgersEulerSampler(rw, n_steps=n_steps)
        x_pred = sampler.sample(c)
        J, E = compute_J_and_energy(x_pred, c)
        elapsed = time.time() - t0
        results[g] = {"J": J, "Energy": E, "wall_s": elapsed}
        b = DDPM_BASELINE_FOPC[g]
        pbar.set_postfix({"γ": g, "FM J": f"{J:.4f}"})
        tqdm.write(f"{g:>5.1f} | {J:>10.5f} | {b['J']:>10.5f} | {E:>10.1f} | {b['Energy']:>12.1f} | {elapsed:>8.1f}s")
    print("=" * 92)

    if save_path is None:
        save_path = os.path.join(ROOT, "flow", "lab_four_gamma_sweep.png")

    try:
        import matplotlib.pyplot as plt
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        gs = list(results.keys())
        axes[0].plot(gs, [results[g]["J"] for g in gs], "o-", label="FM (ours)")
        axes[0].plot(gs, [DDPM_BASELINE_FOPC[g]["J"] for g in gs], "s--", label="DDPM (baseline)")
        axes[0].set_xlabel("γ"); axes[0].set_ylabel("J"); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot(gs, [results[g]["Energy"] for g in gs], "o-", label="FM")
        axes[1].plot(gs, [DDPM_BASELINE_FOPC[g]["Energy"] for g in gs], "s--", label="DDPM")
        axes[1].set_xlabel("γ"); axes[1].set_ylabel("Energy"); axes[1].legend(); axes[1].grid(alpha=0.3)
        fig.suptitle(f"FM vs DDPM, n_samples={n_samples}, n_steps={n_steps}, seed={seed}")
        plt.tight_layout()
        os.makedirs(os.path.dirname(save_path) or ".", exist_ok=True)
        plt.savefig(save_path, dpi=120, bbox_inches="tight")
        print(f"\nSaved plot: {save_path}")
        plt.close()
    except Exception as e:
        print(f"(plot failed: {e})")

    return results

# Sanity Checks

Run these in order after filling in each Part:

1. `sanity_check_part1()` — visualize Burgers data (no fill-in needed; already runs)
2. `sanity_check_2_4()` — after Q2.1-Q2.3: joint FM trains and samples (loss drops, output not noise)
3. `sanity_check_3_3()` — after Q3.1-Q3.2: sample with γ=1 and check J
4. `sanity_check_4_5()` — after Q4.1-Q4.4: γ=1 reweighted == joint output, γ≠1 differs

In [36]:
def sanity_check_part1():
    """Just load data and visualize — no fill-ins needed."""
    print("\n=== Sanity Check Part 1: visualize Burgers data ===")
    ds = load_burgers_train()
    print(f"  dataset has {len(ds)} samples; one sample shape = {ds[0].shape}")
    x = torch.stack([ds[i] for i in range(3)])
    visualize_trajectory(x, "Burgers training data — 3 samples",
                          save_path=os.path.join(HERE, "lab_four_part1_data.png"))
    visualize_noisy_samples(x[:1],
                            save_path=os.path.join(HERE, "lab_four_part1_noisy.png"))

In [37]:
def sanity_check_2_4(num_steps: int = 500, batch_size: int = 32):
    """After filling Q2.1-Q2.3: train the joint FM model for a few hundred steps
    and verify (a) loss curve goes down, (b) sample output is not pure noise."""
    print("\n=== Sanity Check 2.4: joint FM trains and samples ===")
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"  device: {device}")

    ds = BurgersDataset(load_burgers_train(device=device), device=device)
    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    net = BurgersVectorField(dim=32, dim_mults=(1, 2, 4)).to(device)
    trainer = BurgersFlowTrainer(net, path, ds, lr=1e-3)

    losses = trainer.train(num_steps=num_steps, batch_size=batch_size, print_every=50)

    # Sample one trajectory and visualize (should not look like pure noise)
    _, c_test = ds.sample(1)
    sampler = BurgersEulerSampler(net, n_steps=50)
    x_pred = sampler.sample(c_test)
    print(f"  predicted trajectory range: u in [{x_pred[0,0,:11].min():.2f}, {x_pred[0,0,:11].max():.2f}]")
    print(f"  predicted trajectory range: w in [{x_pred[0,1,:11].min():.2f}, {x_pred[0,1,:11].max():.2f}]")

    # If loss went down, we're good
    early = float(np.mean(losses[:20]))
    late = float(np.mean(losses[-20:]))
    print(f"  loss early avg = {early:.4f}, late avg = {late:.4f}")
    if late < early * 0.5:
        print("  ✅ Loss dropped > 2x — joint FM training is working.")
    else:
        print("  ⚠️  Loss didn't drop much; check Q2.2 implementation.")

In [38]:
def sanity_check_3_3(num_train_steps: int = 300, n_sample_steps: int = 50):
    """After filling Q3.1, Q3.2: train small net, sample with γ=1, visualize, compute J.

    Self-contained — trains a small net inside so you don't need to wire one in
    from sanity_check_2_4. Fast sanity, NOT paper quality.

    NOTE: viz is now 4-panel (mirrors plot_inference.py): pred u | pred w | sim u | terminal.
    """
    print("\n=== Sanity Check 3.3: sample with γ=1 and compute J ===")
    device = "mps" if torch.backends.mps.is_available() else "cpu"
    print(f"  device: {device}")

    ds = BurgersDataset(load_burgers_train(device=device), device=device)
    path = GaussianConditionalProbabilityPath(LinearAlpha(), LinearBeta())
    net = BurgersVectorField(dim=32, dim_mults=(1, 2, 4)).to(device)
    trainer = BurgersFlowTrainer(net, path, ds, lr=1e-3)
    print(f"  training {num_train_steps} steps (small net, ~30s on M4 Pro)...")
    trainer.train(num_steps=num_train_steps, batch_size=32, print_every=100)

    net.eval()
    sampler = BurgersEulerSampler(net, n_steps=n_sample_steps)
    _, c = ds.sample(3)
    x_pred = sampler.sample(c)
    print(f"  sampled shape: {tuple(x_pred.shape)}")

    J, E = compute_J_and_energy(x_pred, c)
    baseline_J = 0.0082
    baseline_E = 1656
    print(f"  J      = {J:.4f}   (DDPM baseline γ=1: {baseline_J})")
    print(f"  Energy = {E:.1f}     (DDPM baseline γ=1: {baseline_E})")
    print(f"  J / baseline = {J / baseline_J:.1f}x")
    if J < baseline_J * 50:
        print(f"  ✅ Within 50x of baseline — Q3 pipeline works (small net, {num_train_steps} train steps).")
    else:
        print(f"  ⚠️  More than 50x worse than baseline — check Q3.1/Q3.2 fill-ins.")

    visualize_trajectory_with_simulation(
        x_pred, c,
        title=f"Sampled trajectories (γ=1, joint only, {num_train_steps} train steps)",
        save_path=os.path.join(HERE, "lab_four_part3_sample.png"),
    )
    return net

In [39]:
def sanity_check_4_5(net_joint, net_prior, device: str = "cpu"):
    """After filling Q4.1-Q4.4: verify γ=1 reweighted == joint output exactly, and γ≠1 differs."""
    print("\n=== Sanity Check 4.5: γ=1 reduces to joint, γ≠1 differs ===")
    ds = BurgersDataset(load_burgers_train(device=device), device=device)
    _, c = ds.sample(2)
    x = torch.randn(2, 2, 16, 128, device=device)
    t = torch.full((2,), 0.5, device=device)

    rw_g1  = ReweightedVectorField(net_joint, net_prior, gamma=1.0, use_scheduler=True).to(device).eval()
    rw_g25 = ReweightedVectorField(net_joint, net_prior, gamma=2.5, use_scheduler=True).to(device).eval()

    with torch.no_grad():
        v_g1  = rw_g1(x, t, c)
        v_g25 = rw_g25(x, t, c)
        v_joint = net_joint(x, t, c)

    err_g1 = (v_g1 - v_joint).abs().max().item()
    diff_g25 = (v_g25 - v_joint).abs().max().item()
    print(f"  ||v_γ=1 - v_joint||_∞  = {err_g1:.6e}  (should be ≈ 0)")
    print(f"  ||v_γ=2.5 - v_joint||_∞ = {diff_g25:.6e}  (should be > 0)")
    if err_g1 < 1e-5 and diff_g25 > 1e-4:
        print("  ✅ γ-reweighting math checks out.")
    else:
        print("  ⚠️  Something off; double-check Q4.4 formula and signs.")

# Main entry point

The `if __name__ == "__main__"` block below just runs `sanity_check_part1()` and prints what to do next.

You won't typically run this cell in the notebook — just call the individual `sanity_check_*` functions as you finish each Part.

In [40]:
if __name__ == "__main__":
    print(__doc__)
    print("\n>>> Running Part 1 sanity (no fill-in needed):")
    # sanity_check_part1()
    # sanity_check_2_4()
    
    print(">>> Once Q4.1-Q4.4 are filled, run:  sanity_check_4_5(net_joint, net_prior)")
    print(">>> When all sanity checks pass, run: part5_gamma_sweep(net_joint, net_prior)")

Automatically created module for IPython interactive environment

>>> Running Part 1 sanity (no fill-in needed):
>>> Once Q4.1-Q4.4 are filled, run:  sanity_check_4_5(net_joint, net_prior)
>>> When all sanity checks pass, run: part5_gamma_sweep(net_joint, net_prior)


In [41]:
# sanity_check_3_3(num_train_steps=200, n_sample_steps=5000)

In [42]:
# device = "mps" if torch.backends.mps.is_available() else "cpu"
# net_joint = BurgersVectorField(dim=32, dim_mults=(1, 2, 4)).to(device)
# net_prior = BurgersVectorField(dim=32, dim_mults=(1, 2, 4)).to(device)

# sanity_check_4_5(net_joint, net_prior, device=device)

In [43]:
import os
os.makedirs("flow/checkpoints", exist_ok=True)
# 小规模训练(对标 DDPM 配置但 1/5 步数)

net_joint = train_joint_for_part5(
      num_steps=25000,
      checkpoint_every=2500,    # ← 每 100 步存一次 ckpt + 刷新 loss 图
      save_path="flow/checkpoints/fm_joint.pt",
  )
# 预计 30-40 min on M4 Pro
# 期望:loss 从 ~1.0 单调下降到 ~0.01-0.02 范围



net_prior = train_prior_for_part5(
      num_steps=6250,
      checkpoint_every=1250,    # ← 每 100 步存一次 ckpt + 刷新 loss 图
      save_path="flow/checkpoints/fm_prior.pt",
  )
# 预计 10-15 min on M4 Pro
# 期望:loss 同样下降到 ~0.01-0.02



/Users/baochen/diffphycon/dataset/apps/burgers_h5py.py:124: UserWarning: 'makedirs' is deprecated, use 'os.makedirs(path, exist_ok=True)' instead
  makedirs(self.processed_dir)


  device: mps
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  dataset size: 8000 samples (vs sanity's 160)
  net params: 35,707,906
  training 25000 steps (checkpoint every 2500)...


train: 100%|██████████| 2500/2500 [25:12<00:00,  1.65it/s, loss=0.0073, avg200=0.0097]


  saved loss plot: flow/checkpoints/fm_joint_losses.png
  [checkpoint] step 2500/25000  →  flow/checkpoints/fm_joint_step2500.pt


train: 100%|██████████| 2500/2500 [25:23<00:00,  1.64it/s, loss=0.0068, avg200=0.0066]


...

  saved loss plot: flow/checkpoints/fm_prior_losses.png
  [checkpoint] step 3750/6250  →  flow/checkpoints/fm_prior_step3750.pt


train: 100%|██████████| 1250/1250 [12:32<00:00,  1.66it/s, loss=0.0045, avg200=0.0047]


  saved loss plot: flow/checkpoints/fm_prior_losses.png
  [checkpoint] step 5000/6250  →  flow/checkpoints/fm_prior_step5000.pt


train: 100%|██████████| 1250/1250 [12:32<00:00,  1.66it/s, loss=0.0038, avg200=0.0045]


  saved loss plot: flow/checkpoints/fm_prior_losses.png
  [checkpoint] step 6250/6250  →  flow/checkpoints/fm_prior_step6250.pt
  done. final loss (smoothed): 0.00460


In [44]:
for g in [0.3, 0.5, 0.7, 0.9, 1.0, 1.5, 2.5]:
      inference_and_plot(net_joint, net_prior, gamma=g,
                         save_path=f"flow/lab_four_inf_gamma{g}.png")

Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  inference: reweighted with γ=0.3, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


100%|██████████| 10000/10000 [00:00<00:00, 12003.29it/s]


  J      = 0.0011   (DDPM baseline γ=1: 0.0082)  →  0.1x
  Energy = 838.2     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 12014.22it/s]


  saved flow/lab_four_inf_gamma0.3.png
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5


...

  saved flow/lab_four_inf_gamma1.5.png
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


100%|██████████| 10000/10000 [00:00<00:00, 12199.67it/s]


  J      = 0.0008   (DDPM baseline γ=1: 0.0082)  →  0.1x
  Energy = 577.9     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 12138.75it/s]


  saved flow/lab_four_inf_gamma2.5.png


In [45]:
part5_gamma_sweep(net_joint, net_prior, n_samples=8, n_steps=100)

Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5

    γ |       FM J |     DDPM J |  FM Energy |  DDPM Energy |  wall (s)
--------------------------------------------------------------------------------------------


γ sweep:  14%|█▍        | 1/7 [00:07<00:43,  7.23s/it, γ=0.3, FM J=0.0021]

  0.3 |    0.00208 |    0.00830 |      797.2 |       1670.7 |      7.2s


γ sweep:  29%|██▊       | 2/7 [00:14<00:35,  7.16s/it, γ=0.5, FM J=0.0023]

  0.5 |    0.00226 |    0.00828 |      857.4 |       1666.0 |      7.1s


...

γ sweep:  86%|████████▌ | 6/7 [00:42<00:07,  7.10s/it, γ=1.5, FM J=0.0007]

  1.5 |    0.00074 |    0.00811 |      829.1 |       1648.1 |      7.1s


γ sweep: 100%|██████████| 7/7 [00:49<00:00,  7.13s/it, γ=2.5, FM J=0.0013]


  2.5 |    0.00132 |    0.00796 |      865.9 |       1634.0 |      7.1s

Saved plot: /Users/baochen/diffphycon/flow/lab_four_gamma_sweep.png


{0.3: {'J': 0.0020774872973561287,
  'Energy': 797.1551513671875,
  'wall_s': 7.23328709602356},
 0.5: {'J': 0.0022594465408474207,
  'Energy': 857.4490356445312,
  'wall_s': 7.104844093322754},
 0.7: {'J': 0.0007392525440081954,
  'Energy': 862.2724609375,
  'wall_s': 7.129947900772095},
 0.9: {'J': 0.002315951744094491,
  'Energy': 844.142333984375,
  'wall_s': 7.125293016433716},
 1.0: {'J': 0.0029912700410932302,
  'Energy': 812.0119018554688,
  'wall_s': 7.0735578536987305},
 1.5: {'J': 0.0007449293043464422,
  'Energy': 829.0750732421875,
  'wall_s': 7.079571008682251},
 2.5: {'J': 0.001322598080150783,
  'Energy': 865.880615234375,
  'wall_s': 7.127681016921997}}

In [51]:
inference_and_plot(net_joint, net_prior, gamma=2.5, n_steps=50,
                     save_path="flow/lab_four_inf_gamma2.5_50step.png")

Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


100%|██████████| 10000/10000 [00:00<00:00, 12406.41it/s]


  J      = 0.0015   (DDPM baseline γ=1: 0.0082)  →  0.2x
  Energy = 1687.4     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 11849.47it/s]


  saved flow/lab_four_inf_gamma2.5_50step.png


(tensor([[[[ 7.9991e-04,  9.7494e-04,  1.1809e-03,  ..., -1.6463e-02,
            -1.4897e-02, -1.3442e-02],
           [-3.6480e-03, -7.8681e-03, -1.0035e-02,  ..., -7.9750e-03,
            -3.6602e-03, -1.2634e-03],
           [-9.0841e-03, -1.6838e-02, -1.8003e-02,  ..., -1.7945e-03,
            -1.4510e-03,  2.6173e-03],
           ...,
           [-1.7243e-03, -4.0067e-03, -3.7427e-03,  ..., -2.8692e-03,
             5.4856e-04, -1.9883e-03],
           [-1.3706e-03, -2.9640e-03, -1.5364e-03,  ..., -1.9366e-03,
            -2.7664e-03, -2.8373e-03],
           [-1.4889e-03, -3.1296e-03, -2.2728e-04,  ..., -1.9032e-03,
            -2.0317e-03, -1.3545e-03]],
 
          [[-9.3552e-02, -1.0288e-01, -1.0771e-01,  ...,  3.2015e-02,
             3.2407e-02,  3.2376e-02],
           [-1.5328e-01, -1.6282e-01, -1.6927e-01,  ...,  8.5137e-02,
             8.1617e-02,  8.2935e-02],
           [-1.7438e-01, -1.8657e-01, -1.9652e-01,  ...,  1.5026e-01,
             1.4786e-01,  1.4570e-01],


In [66]:
import torch
device = "mps"

def load_ckpt(path, device="mps"):
    net = BurgersVectorField(dim=64, dim_mults=(1, 2, 4, 8)).to(device)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    net.load_state_dict(ckpt['state_dict']); net.eval()
    return net

net_joint = load_ckpt("flow/checkpoints/fm_joint_step25000.pt")
net_prior = load_ckpt("flow/checkpoints/fm_prior_step6250.pt")



part5_gamma_sweep(net_joint, net_prior, n_samples=8, n_steps=100, split="test")
inference_and_plot(net_joint, net_prior, gamma=2.5, split="train",
                    save_path="flow/lab_four_inf_gamma2.5_train.png", n_steps=100)
inference_and_plot(net_joint, net_prior, gamma=2.5, split="test",
                    save_path="flow/lab_four_inf_gamma2.5_test.png", n_steps=100)

Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_test.h5
  using split='test' (2000 samples available)

    γ |       FM J |     DDPM J |  FM Energy |  DDPM Energy |  wall (s)
--------------------------------------------------------------------------------------------


γ sweep:  14%|█▍        | 1/7 [00:07<00:43,  7.26s/it, γ=0.3, FM J=0.0028]

  0.3 |    0.00284 |    0.00830 |      566.9 |       1670.7 |      7.3s


γ sweep:  29%|██▊       | 2/7 [00:14<00:36,  7.24s/it, γ=0.5, FM J=0.0020]

  0.5 |    0.00199 |    0.00828 |      593.5 |       1666.0 |      7.2s


...

100%|██████████| 10000/10000 [00:00<00:00, 11923.28it/s]


  J      = 0.0013   (DDPM baseline γ=1: 0.0082)  →  0.2x
  Energy = 1290.0     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 11816.71it/s]


  saved flow/lab_four_inf_gamma2.5_test.png


(tensor([[[[ 2.7373e-04,  3.1757e-04,  3.6662e-04,  ..., -1.3052e-02,
            -1.1365e-02, -9.8407e-03],
           [ 9.1769e-03,  4.2356e-03,  3.9097e-03,  ..., -1.3620e-02,
            -1.5009e-02,  8.1274e-04],
           [ 5.4897e-04,  1.2361e-02,  5.1856e-03,  ..., -7.7615e-03,
             5.8885e-04, -1.5595e-02],
           ...,
           [ 1.8369e-03, -1.1987e-02, -1.3046e-02,  ...,  6.5387e-03,
            -5.4186e-04,  1.5970e-03],
           [-4.0898e-03,  3.6612e-03, -2.3664e-03,  ...,  4.5758e-03,
            -4.3171e-04, -6.4731e-04],
           [ 5.2196e-03, -7.5315e-03, -2.3364e-03,  ..., -6.0867e-04,
            -3.5916e-03,  2.3977e-03]],
 
          [[ 1.7739e-02,  2.3207e-02,  3.3667e-02,  ..., -1.0489e-02,
             7.5113e-03, -1.2287e-02],
           [ 7.4588e-02,  7.6674e-02,  7.6103e-02,  ...,  3.8944e-03,
             2.7578e-03,  1.4162e-03],
           [ 1.1358e-01,  1.2601e-01,  1.4377e-01,  ..., -1.4500e-02,
             8.2492e-03,  8.8033e-03],


In [81]:
for n in [100, 500, 1000]:
    print(f"\n========== n_steps={n}, seed=42 ==========")
    part5_gamma_sweep(
        net_joint, net_prior,
        n_samples=8, n_steps=n, seed=42,
        save_path=f"/Users/baochen/diffphycon/flow/lab_four_gamma_sweep_n{n}_seed42.png",)


========== n_steps=100, seed=42 ==========
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  using split='train' (8000 samples available), seed=42

    γ |       FM J |     DDPM J |  FM Energy |  DDPM Energy |  wall (s)
--------------------------------------------------------------------------------------------


γ sweep:  14%|█▍        | 1/7 [00:07<00:46,  7.73s/it, γ=0.3, FM J=0.0044]

  0.3 |    0.00440 |    0.00830 |      463.6 |       1670.7 |      7.7s


γ sweep:  29%|██▊       | 2/7 [00:14<00:37,  7.42s/it, γ=0.5, FM J=0.0042]

  0.5 |    0.00420 |    0.00828 |      466.4 |       1666.0 |      7.2s


...

  1.0 |    0.00300 |    0.00820 |      454.5 |       1656.2 |     59.9s


γ sweep:  86%|████████▌ | 6/7 [05:59<00:59, 59.97s/it, γ=1.5, FM J=0.0040]

  1.5 |    0.00396 |    0.00811 |      455.7 |       1648.1 |     60.1s


γ sweep: 100%|██████████| 7/7 [06:59<00:00, 59.99s/it, γ=2.5, FM J=0.0034]


  2.5 |    0.00339 |    0.00796 |      438.1 |       1634.0 |     60.4s

Saved plot: /Users/baochen/diffphycon/flow/lab_four_gamma_sweep_n1000_seed42.png


In [ ]:
compare_n_steps_visually(
      net_joint, net_prior,
      gamma=2.5,
      n_steps_list=[100, 500, 1000],
      seed=42,
  )


=== compare_n_steps_visually: γ=2.5, seed=42 ===

--- n_steps=100 ---
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  dataset split: train (8000 samples), seed=42
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


100%|██████████| 10000/10000 [00:00<00:00, 11754.69it/s]


  J      = 0.0092   (DDPM baseline γ=1: 0.0082)  →  1.1x
  Energy = 307.9     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 11934.04it/s]


  saved /Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n100_seed42.png

--- n_steps=500 ---
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  dataset split: train (8000 samples), seed=42
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


...

100%|██████████| 10000/10000 [00:00<00:00, 11883.57it/s]


  J      = 0.0093   (DDPM baseline γ=1: 0.0082)  →  1.1x
  Energy = 305.1     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 11917.93it/s]


  saved /Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n1000_seed42.png

--- summary ---
 n_steps |          J |     Energy
     100 |    0.00915 |      307.9
     500 |    0.00934 |      306.0
    1000 |    0.00932 |      305.1


[{'n_steps': 100,
  'J': 0.009154441766440868,
  'Energy': 307.884033203125,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n100_seed42.png'},
 {'n_steps': 500,
  'J': 0.009335723705589771,
  'Energy': 306.04931640625,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n500_seed42.png'},
 {'n_steps': 1000,
  'J': 0.009320954792201519,
  'Energy': 305.1350402832031,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n1000_seed42.png'}]

In [91]:
import torch
device = "mps"

def load_ckpt(path):
    net = BurgersVectorField(dim=64, dim_mults=(1, 2, 4, 8)).to(device)
    ckpt = torch.load(path, map_location=device, weights_only=False)
    net.load_state_dict(ckpt['state_dict']); net.eval()
    return net

# 1. 載入之前訓練好的權重
net_joint = load_ckpt("flow/checkpoints/fm_joint_ema.pt")
net_prior = load_ckpt("flow/checkpoints/fm_prior_ema.pt")

# 2. 【修改這裡】指定更大的數據集名稱 (1e5)
LARGE_DATASET = "free_u_f_1e5_front_rear_quarter"

# 先載入底層的大數據集
ds_full = load_burgers_train(device=device, dataset=LARGE_DATASET)

# 再分別包裝給 Joint 和 Prior 使用
ds_train = BurgersDataset(ds_full, device=device)
ds_prior = BurgersPriorDataset(ds_full, device=device)

# 3. 使用 finetune_with_ema 接著訓練 (你可以根據需要調大 num_steps)
# 假設我們在大數據集上再多跑 10,000 步
net_joint_ema = finetune_with_ema(
    net_joint, BurgersFlowTrainer, ds_train,
    num_steps=10000, ema_decay=0.995,
    save_path_ema="flow/checkpoints/fm_joint_ema_large.pt",
)

# Prior 模型的步數通常是 Joint 的 1/4 即可
net_prior_ema = finetune_with_ema(
    net_prior, BurgersPriorTrainer, ds_prior,
    num_steps=6000, ema_decay=0.995,
    save_path_ema="flow/checkpoints/fm_prior_ema_large.pt",
)

Load dataset /Users/baochen/diffphycon/data/free_u_f_1e5_front_rear_quarter/burgers_train.h5
  fine-tuning 10000 steps with EMA decay=0.995...


finetune+EMA: 100%|██████████| 10000/10000 [1:44:15<00:00,  1.60it/s, loss=0.0006, avg100=0.0011]


  saved EMA-weighted net: flow/checkpoints/fm_joint_ema_large.pt
  done. final loss (smoothed): 0.00108
  fine-tuning 6000 steps with EMA decay=0.995...


finetune+EMA: 100%|██████████| 6000/6000 [1:01:16<00:00,  1.63it/s, loss=0.0008, avg100=0.0008]


  saved EMA-weighted net: flow/checkpoints/fm_prior_ema_large.pt
  done. final loss (smoothed): 0.00078


In [92]:
compare_n_steps_visually(
      net_joint_ema, net_prior_ema,
      gamma=2.5,
      n_steps_list=[100, 500, 1000],
      seed=42,
  )


=== compare_n_steps_visually: γ=2.5, seed=42 ===

--- n_steps=100 ---
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  dataset split: train (8000 samples), seed=42
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


100%|██████████| 10000/10000 [00:00<00:00, 12556.14it/s]


  J      = 0.0122   (DDPM baseline γ=1: 0.0082)  →  1.5x
  Energy = 282.4     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 12288.91it/s]


  saved /Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n100_seed42.png

--- n_steps=500 ---
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5
  dataset split: train (8000 samples), seed=42
  inference: reweighted with γ=2.5, scheduler=True
  sampled 3 trajectories, shape=(3, 2, 16, 128)


...

100%|██████████| 10000/10000 [00:00<00:00, 11723.51it/s]


  J      = 0.0123   (DDPM baseline γ=1: 0.0082)  →  1.5x
  Energy = 278.7     (DDPM baseline γ=1: 1656)


100%|██████████| 10000/10000 [00:00<00:00, 11737.92it/s]


  saved /Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n1000_seed42.png

--- summary ---
 n_steps |          J |     Energy
     100 |    0.01217 |      282.4
     500 |    0.01233 |      279.1
    1000 |    0.01234 |      278.7


[{'n_steps': 100,
  'J': 0.01217483077198267,
  'Energy': 282.38482666015625,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n100_seed42.png'},
 {'n_steps': 500,
  'J': 0.0123292850330472,
  'Energy': 279.0739440917969,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n500_seed42.png'},
 {'n_steps': 1000,
  'J': 0.012340407818555832,
  'Energy': 278.7132873535156,
  'path': '/Users/baochen/diffphycon/flow/lab_four_inf_gamma2.5_n1000_seed42.png'}]

In [87]:
for n in [100, 500, 1000]:
    print(f"\n========== n_steps={n}, seed=42 ==========")
    part5_gamma_sweep(
        net_joint_ema, net_prior_ema,
        n_samples=8, n_steps=n, seed=42,
        save_path=f"/Users/baochen/diffphycon/flow/lab_four_gamma_sweep_n{n}_seed42.png",)


========== n_steps=100, seed=42 ==========
Load dataset /Users/baochen/diffphycon/data/free_u_f_1e4_front_rear_quarter/burgers_train.h5


/Users/baochen/diffphycon/dataset/apps/burgers_h5py.py:124: UserWarning: 'makedirs' is deprecated, use 'os.makedirs(path, exist_ok=True)' instead
  makedirs(self.processed_dir)


  using split='train' (8000 samples available), seed=42

    γ |       FM J |     DDPM J |  FM Energy |  DDPM Energy |  wall (s)
--------------------------------------------------------------------------------------------


γ sweep:  14%|█▍        | 1/7 [00:08<00:48,  8.11s/it, γ=0.3, FM J=0.0034]

  0.3 |    0.00342 |    0.00830 |      492.8 |       1670.7 |      8.1s


...

  1.0 |    0.00309 |    0.00820 |      495.1 |       1656.2 |     61.5s


γ sweep:  86%|████████▌ | 6/7 [06:08<01:01, 61.51s/it, γ=1.5, FM J=0.0030]

  1.5 |    0.00302 |    0.00811 |      500.1 |       1648.1 |     61.5s


γ sweep: 100%|██████████| 7/7 [07:09<00:00, 61.41s/it, γ=2.5, FM J=0.0027]


  2.5 |    0.00270 |    0.00796 |      481.8 |       1634.0 |     61.5s

Saved plot: /Users/baochen/diffphycon/flow/lab_four_gamma_sweep_n1000_seed42.png
